## 0. 公共定义（波形、六种协议步序、演化函数）

In [ ]:
import numpy as np
if not hasattr(np, 'trapz'):
    np.trapz = np.trapezoid
from qutip import *
import matplotlib.pyplot as plt
from tqdm import tqdm
import itertools

import qutip as _qt
_QT5 = int(_qt.__version__.split('.')[0]) >= 5
def solver_opts(T_val):
    od = dict(nsteps=100000, atol=1e-10, rtol=1e-8, max_step=T_val/10)
    return od if _QT5 else Options(**od)

np.seterr(divide='ignore', invalid='ignore')

dim = 4
N_per_step = 1000
T_values = np.arange(0.5, 30.5, 0.5)

a = basis(6, 0)
b = basis(6, 1)
c = basis(6, 2)
d = basis(6, 3)
e = basis(6, 4)
f = basis(6, 5)

F4 = 0.5 * np.array([[1, 1, 1, 1],
                     [1, 1j, -1, -1j],
                     [1, -1, 1, -1],
                     [1, -1j, -1, 1j]])

def stokes_op(phi):
    return a * b.dag() * np.conj(np.exp(1j*phi)) + b * a.dag() * np.exp(1j*phi)

def make_coeff_from_arr(arr):
    def func(t, args):
        return np.interp(t, args['tlist'], arr)
    return func

def get_STIRAP_waveforms(tstep, tau, amps, ampp_list, psi, T, eps=1e-12):
    S_half = 0.5 * amps * np.exp(-(tstep + tau + psi)**2 / T**2)
    P_half_list = [0.5 * amp * np.exp(-(tstep + psi)**2 / T**2) for amp in ampp_list]
    return S_half, P_half_list, []

def get_all_optical_waveforms(tstep, tau, amps, ampp_list, psi, T, eps=1e-12):
    dt = tstep[1] - tstep[0]
    S_half_raw = 0.5 * amps * np.exp(-(tstep + tau + psi)**2 / T**2)
    P_half_raw_list = [0.5 * amp * np.exp(-(tstep + psi)**2 / T**2) for amp in ampp_list]

    Omega_S = 2.0 * S_half_raw
    Omega_P_list = [2.0 * p for p in P_half_raw_list]
    Omega_P = np.sqrt(sum(p**2 for p in Omega_P_list))

    dOmega_P = np.gradient(Omega_P, dt)
    dOmega_S = np.gradient(Omega_S, dt)

    denom = Omega_P**2 + Omega_S**2
    theta_dot = np.divide(dOmega_P * Omega_S - dOmega_S * Omega_P,
                          denom,
                          where=(denom >= eps),
                          out=np.zeros_like(Omega_S))

    sigma_sg = 3.0
    p_sg = 10
    tc = -psi
    window = np.exp(-(np.abs(tstep - tc) / (sigma_sg * T))**p_sg)
    Omega_a = 2.0 * theta_dot * window

    phi_rot = np.arctan2(Omega_a, Omega_P)
    dphi_rot = np.gradient(phi_rot, dt)

    Omega_P_tilde = np.sqrt(Omega_P**2 + Omega_a**2)
    Omega_S_tilde = Omega_S - 2.0 * dphi_rot

    scale = np.ones_like(Omega_P)
    mask_P = Omega_P > eps
    scale[mask_P] = Omega_P_tilde[mask_P] / Omega_P[mask_P]

    S_half_mod = Omega_S_tilde / 2.0
    P_half_mod_list = [0.5 * p * scale for p in Omega_P_list]
    return S_half_mod, P_half_mod_list, []

def get_STIRSAP_waveforms(tstep, tau, amps, ampp_list, psi, T, eps=1e-12):
    dt = tstep[1] - tstep[0]
    S_half = 0.5 * amps * np.exp(-(tstep + tau + psi)**2 / T**2)
    P_half_raw_list = [0.5 * amp * np.exp(-(tstep + psi)**2 / T**2) for amp in ampp_list]

    Omega_S = 2.0 * S_half
    Omega_P_list = [2.0 * p for p in P_half_raw_list]
    Omega_P = np.sqrt(sum(p**2 for p in Omega_P_list))

    dOmega_P = np.gradient(Omega_P, dt)
    dOmega_S = np.gradient(Omega_S, dt)

    denom = Omega_P**2 + Omega_S**2
    theta_dot = np.divide(dOmega_P * Omega_S - dOmega_S * Omega_P,
                          denom,
                          where=(denom >= eps),
                          out=np.zeros_like(Omega_S))
    Omega_cd = 2.0 * theta_dot

    sigma_sg = 3.0
    p_sg = 10
    tc = -psi
    window = np.exp(-(np.abs(tstep - tc) / (sigma_sg * T))**p_sg)
    Omega_cd = Omega_cd * window

    scale = np.zeros_like(Omega_P)
    mask_P = Omega_P > eps
    scale[mask_P] = Omega_cd[mask_P] / Omega_P[mask_P]

    cd_half_list = [0.5 * p * scale for p in Omega_P_list]
    return S_half, P_half_raw_list, cd_half_list

qubit_STIRAP_steps = [
    {'phi': 0, 'tau_factor': 1, 'amps': 2.0, 'ampp_list': [np.sqrt(2 - np.sqrt(2)), np.sqrt(2 + np.sqrt(2))], 'pump_ops': [-b*d.dag()-d*b.dag(), b*f.dag()+f*b.dag()]},
    {'phi': np.pi, 'tau_factor': -1, 'amps': 2.0, 'ampp_list': [np.sqrt(2 - np.sqrt(2)), np.sqrt(2 + np.sqrt(2))], 'pump_ops': [-b*d.dag()-d*b.dag(), b*f.dag()+f*b.dag()]},
    {'phi': 0, 'tau_factor': 1, 'amps': 2.0, 'ampp_list': [np.sqrt(2 - np.sqrt(2)), np.sqrt(2 + np.sqrt(2))], 'pump_ops': [-b*c.dag()-c*b.dag(), b*e.dag()+e*b.dag()]},
    {'phi': np.pi, 'tau_factor': -1, 'amps': 2.0, 'ampp_list': [np.sqrt(2 - np.sqrt(2)), np.sqrt(2 + np.sqrt(2))], 'pump_ops': [-b*c.dag()-c*b.dag(), b*e.dag()+e*b.dag()]},
    {'phi': 0, 'tau_factor': 1, 'amps': 2.0, 'ampp_list': [2.0], 'pump_ops': [b*f.dag()+f*b.dag()]},
    {'phi': 0.5*np.pi, 'tau_factor': -1, 'amps': 2.0, 'ampp_list': [2.0], 'pump_ops': [b*f.dag()+f*b.dag()]},
    {'phi': 0, 'tau_factor': 1, 'amps': 2.0, 'ampp_list': [np.sqrt(2 - np.sqrt(2)), np.sqrt(2 + np.sqrt(2))], 'pump_ops': [b*e.dag()+e*b.dag(), -b*f.dag()-f*b.dag()]},
    {'phi': np.pi, 'tau_factor': -1, 'amps': 2.0, 'ampp_list': [np.sqrt(2 - np.sqrt(2)), np.sqrt(2 + np.sqrt(2))], 'pump_ops': [b*e.dag()+e*b.dag(), -b*f.dag()-f*b.dag()]},
    {'phi': 0, 'tau_factor': 1, 'amps': 2.0, 'ampp_list': [np.sqrt(2 - np.sqrt(2)), np.sqrt(2 + np.sqrt(2))], 'pump_ops': [b*c.dag()+c*b.dag(), -b*d.dag()-d*b.dag()]},
    {'phi': np.pi, 'tau_factor': -1, 'amps': 2.0, 'ampp_list': [np.sqrt(2 - np.sqrt(2)), np.sqrt(2 + np.sqrt(2))], 'pump_ops': [b*c.dag()+c*b.dag(), -b*d.dag()-d*b.dag()]},
    {'phi': 0, 'tau_factor': 1, 'amps': 2.0, 'ampp_list': [np.sqrt(2), np.sqrt(2)], 'pump_ops': [-b*d.dag()-d*b.dag(), b*e.dag()+e*b.dag()]},
    {'phi': np.pi, 'tau_factor': -1, 'amps': 2.0, 'ampp_list': [np.sqrt(2), np.sqrt(2)], 'pump_ops': [-b*d.dag()-d*b.dag(), b*e.dag()+e*b.dag()]},
]

quartit_STIRAP_steps = [
    {'phi': 0, 'tau_factor': 1, 'amps': 2.0, 'ampp_list': [np.sqrt(2), -np.sqrt(2)], 'pump_ops': [b*d.dag()+d*b.dag(), b*f.dag()+f*b.dag()]},
    {'phi': np.pi/2, 'tau_factor': -1, 'amps': 2.0, 'ampp_list': [np.sqrt(2), -np.sqrt(2)], 'pump_ops': [b*d.dag()+d*b.dag(), b*f.dag()+f*b.dag()]},
    {'phi': 0, 'tau_factor': 1, 'amps': 2.0, 'ampp_list': [-1.0, 1.0, 1.0, 1.0], 'pump_ops': [b*c.dag()+c*b.dag(), b*d.dag()+d*b.dag(), b*e.dag()+e*b.dag(), b*f.dag()+f*b.dag()]},
    {'phi': np.pi, 'tau_factor': -1, 'amps': 2.0, 'ampp_list': [-1.0, 1.0, 1.0, 1.0], 'pump_ops': [b*c.dag()+c*b.dag(), b*d.dag()+d*b.dag(), b*e.dag()+e*b.dag(), b*f.dag()+f*b.dag()]},
]

qubit_alloptical_steps = qubit_STIRAP_steps
quartit_alloptical_steps = quartit_STIRAP_steps

qubit_STIRSAP_steps = [
    {'phi': 0, 'tau_factor': 1, 'amps': 2.0, 'ampp_list': [np.sqrt(2 - np.sqrt(2)), np.sqrt(2 + np.sqrt(2))],
     'pump_ops': [-b*d.dag()-d*b.dag(), b*f.dag()+f*b.dag()],
     'cd_ops': [-a*d.dag()*np.conj(np.exp(1j*(0+0.5*np.pi))) - d*a.dag()*np.exp(1j*(0+0.5*np.pi)), a*f.dag()*np.conj(np.exp(1j*(0+0.5*np.pi))) + f*a.dag()*np.exp(1j*(0+0.5*np.pi))]},
    {'phi': np.pi, 'tau_factor': -1, 'amps': 2.0, 'ampp_list': [np.sqrt(2 - np.sqrt(2)), np.sqrt(2 + np.sqrt(2))],
     'pump_ops': [-b*d.dag()-d*b.dag(), b*f.dag()+f*b.dag()],
     'cd_ops': [-a*d.dag()*np.conj(np.exp(1j*(np.pi+0.5*np.pi))) - d*a.dag()*np.exp(1j*(np.pi+0.5*np.pi)), a*f.dag()*np.conj(np.exp(1j*(np.pi+0.5*np.pi))) + f*a.dag()*np.exp(1j*(np.pi+0.5*np.pi))]},
    {'phi': 0, 'tau_factor': 1, 'amps': 2.0, 'ampp_list': [np.sqrt(2 - np.sqrt(2)), np.sqrt(2 + np.sqrt(2))],
     'pump_ops': [-b*c.dag()-c*b.dag(), b*e.dag()+e*b.dag()],
     'cd_ops': [-a*c.dag()*np.conj(np.exp(1j*(0+0.5*np.pi))) - c*a.dag()*np.exp(1j*(0+0.5*np.pi)), a*e.dag()*np.conj(np.exp(1j*(0+0.5*np.pi))) + e*a.dag()*np.exp(1j*(0+0.5*np.pi))]},
    {'phi': np.pi, 'tau_factor': -1, 'amps': 2.0, 'ampp_list': [np.sqrt(2 - np.sqrt(2)), np.sqrt(2 + np.sqrt(2))],
     'pump_ops': [-b*c.dag()-c*b.dag(), b*e.dag()+e*b.dag()],
     'cd_ops': [-a*c.dag()*np.conj(np.exp(1j*(np.pi+0.5*np.pi))) - c*a.dag()*np.exp(1j*(np.pi+0.5*np.pi)), a*e.dag()*np.conj(np.exp(1j*(np.pi+0.5*np.pi))) + e*a.dag()*np.exp(1j*(np.pi+0.5*np.pi))]},
    {'phi': 0, 'tau_factor': 1, 'amps': 2.0, 'ampp_list': [2.0],
     'pump_ops': [b*f.dag()+f*b.dag()],
     'cd_ops': [a*f.dag()*np.conj(np.exp(1j*(0+0.5*np.pi))) + f*a.dag()*np.exp(1j*(0+0.5*np.pi))]},
    {'phi': 0.5*np.pi, 'tau_factor': -1, 'amps': 2.0, 'ampp_list': [2.0],
     'pump_ops': [b*f.dag()+f*b.dag()],
     'cd_ops': [a*f.dag()*np.conj(np.exp(1j*(0.5*np.pi+0.5*np.pi))) + f*a.dag()*np.exp(1j*(0.5*np.pi+0.5*np.pi))]},
    {'phi': 0, 'tau_factor': 1, 'amps': 2.0, 'ampp_list': [np.sqrt(2 - np.sqrt(2)), np.sqrt(2 + np.sqrt(2))],
     'pump_ops': [b*e.dag()+e*b.dag(), -b*f.dag()-f*b.dag()],
     'cd_ops': [a*e.dag()*np.conj(np.exp(1j*(0+0.5*np.pi))) + e*a.dag()*np.exp(1j*(0+0.5*np.pi)), -a*f.dag()*np.conj(np.exp(1j*(0+0.5*np.pi))) - f*a.dag()*np.exp(1j*(0+0.5*np.pi))]},
    {'phi': np.pi, 'tau_factor': -1, 'amps': 2.0, 'ampp_list': [np.sqrt(2 - np.sqrt(2)), np.sqrt(2 + np.sqrt(2))],
     'pump_ops': [b*e.dag()+e*b.dag(), -b*f.dag()-f*b.dag()],
     'cd_ops': [a*e.dag()*np.conj(np.exp(1j*(np.pi+0.5*np.pi))) + e*a.dag()*np.exp(1j*(np.pi+0.5*np.pi)), -a*f.dag()*np.conj(np.exp(1j*(np.pi+0.5*np.pi))) - f*a.dag()*np.exp(1j*(np.pi+0.5*np.pi))]},
    {'phi': 0, 'tau_factor': 1, 'amps': 2.0, 'ampp_list': [np.sqrt(2 - np.sqrt(2)), np.sqrt(2 + np.sqrt(2))],
     'pump_ops': [b*c.dag()+c*b.dag(), -b*d.dag()-d*b.dag()],
     'cd_ops': [a*c.dag()*np.conj(np.exp(1j*(0+0.5*np.pi))) + c*a.dag()*np.exp(1j*(0+0.5*np.pi)), -a*d.dag()*np.conj(np.exp(1j*(0+0.5*np.pi))) - d*a.dag()*np.exp(1j*(0+0.5*np.pi))]},
    {'phi': np.pi, 'tau_factor': -1, 'amps': 2.0, 'ampp_list': [np.sqrt(2 - np.sqrt(2)), np.sqrt(2 + np.sqrt(2))],
     'pump_ops': [b*c.dag()+c*b.dag(), -b*d.dag()-d*b.dag()],
     'cd_ops': [a*c.dag()*np.conj(np.exp(1j*(np.pi+0.5*np.pi))) + c*a.dag()*np.exp(1j*(np.pi+0.5*np.pi)), -a*d.dag()*np.conj(np.exp(1j*(np.pi+0.5*np.pi))) - d*a.dag()*np.exp(1j*(np.pi+0.5*np.pi))]},
    {'phi': 0, 'tau_factor': 1, 'amps': 2.0, 'ampp_list': [np.sqrt(2), np.sqrt(2)],
     'pump_ops': [-b*d.dag()-d*b.dag(), b*e.dag()+e*b.dag()],
     'cd_ops': [-a*d.dag()*np.conj(np.exp(1j*(0+0.5*np.pi))) - d*a.dag()*np.exp(1j*(0+0.5*np.pi)), a*e.dag()*np.conj(np.exp(1j*(0+0.5*np.pi))) + e*a.dag()*np.exp(1j*(0+0.5*np.pi))]},
    {'phi': np.pi, 'tau_factor': -1, 'amps': 2.0, 'ampp_list': [np.sqrt(2), np.sqrt(2)],
     'pump_ops': [-b*d.dag()-d*b.dag(), b*e.dag()+e*b.dag()],
     'cd_ops': [-a*d.dag()*np.conj(np.exp(1j*(np.pi+0.5*np.pi))) - d*a.dag()*np.exp(1j*(np.pi+0.5*np.pi)), a*e.dag()*np.conj(np.exp(1j*(np.pi+0.5*np.pi))) + e*a.dag()*np.exp(1j*(np.pi+0.5*np.pi))]},
]

quartit_STIRSAP_steps = [
    {'phi': 0, 'tau_factor': 1, 'amps': 2.0, 'ampp_list': [np.sqrt(2), -np.sqrt(2)],
     'pump_ops': [b*d.dag()+d*b.dag(), b*f.dag()+f*b.dag()],
     'cd_ops': [a*d.dag()*np.conj(np.exp(1j*(0+0.5*np.pi))) + d*a.dag()*np.exp(1j*(0+0.5*np.pi)),
                a*f.dag()*np.conj(np.exp(1j*(0+0.5*np.pi))) + f*a.dag()*np.exp(1j*(0+0.5*np.pi))]},
    {'phi': np.pi/2, 'tau_factor': -1, 'amps': 2.0, 'ampp_list': [np.sqrt(2), -np.sqrt(2)],
     'pump_ops': [b*d.dag()+d*b.dag(), b*f.dag()+f*b.dag()],
     'cd_ops': [a*d.dag()*np.conj(np.exp(1j*(np.pi/2+0.5*np.pi))) + d*a.dag()*np.exp(1j*(np.pi/2+0.5*np.pi)),
                a*f.dag()*np.conj(np.exp(1j*(np.pi/2+0.5*np.pi))) + f*a.dag()*np.exp(1j*(np.pi/2+0.5*np.pi))]},
    {'phi': 0, 'tau_factor': 1, 'amps': 2.0, 'ampp_list': [-1.0, 1.0, 1.0, 1.0],
     'pump_ops': [b*c.dag()+c*b.dag(), b*d.dag()+d*b.dag(), b*e.dag()+e*b.dag(), b*f.dag()+f*b.dag()],
     'cd_ops': [a*c.dag()*np.conj(np.exp(1j*(0+0.5*np.pi))) + c*a.dag()*np.exp(1j*(0+0.5*np.pi)),
                a*d.dag()*np.conj(np.exp(1j*(0+0.5*np.pi))) + d*a.dag()*np.exp(1j*(0+0.5*np.pi)),
                a*e.dag()*np.conj(np.exp(1j*(0+0.5*np.pi))) + e*a.dag()*np.exp(1j*(0+0.5*np.pi)),
                a*f.dag()*np.conj(np.exp(1j*(0+0.5*np.pi))) + f*a.dag()*np.exp(1j*(0+0.5*np.pi))]},
    {'phi': np.pi, 'tau_factor': -1, 'amps': 2.0, 'ampp_list': [-1.0, 1.0, 1.0, 1.0],
     'pump_ops': [b*c.dag()+c*b.dag(), b*d.dag()+d*b.dag(), b*e.dag()+e*b.dag(), b*f.dag()+f*b.dag()],
     'cd_ops': [a*c.dag()*np.conj(np.exp(1j*(np.pi+0.5*np.pi))) + c*a.dag()*np.exp(1j*(np.pi+0.5*np.pi)),
                a*d.dag()*np.conj(np.exp(1j*(np.pi+0.5*np.pi))) + d*a.dag()*np.exp(1j*(np.pi+0.5*np.pi)),
                a*e.dag()*np.conj(np.exp(1j*(np.pi+0.5*np.pi))) + e*a.dag()*np.exp(1j*(np.pi+0.5*np.pi)),
                a*f.dag()*np.conj(np.exp(1j*(np.pi+0.5*np.pi))) + f*a.dag()*np.exp(1j*(np.pi+0.5*np.pi))]},
]

def run_protocol(psi0, T, steps, waveform_func):
    psi = psi0
    all_Omega_S = []
    all_Omega_P = []
    all_Omega_CD = []
    t_list_all = []

    for step_idx, step in enumerate(steps):
        phi = step['phi']
        tau = step['tau_factor'] * (T / 0.8)
        amps = step['amps']
        ampp_list = step['ampp_list']
        pump_ops = step['pump_ops']

        psi_shift = -(2*step_idx + 1) * 5 * T
        t_start = step_idx * 10 * T
        t_end = (step_idx + 1) * 10 * T
        tstep = np.linspace(t_start, t_end, N_per_step)

        S_half, P_half_list, cd_half_list = waveform_func(tstep, tau, amps, ampp_list, psi_shift, T)

        all_Omega_S.append(2.0 * S_half)
        all_Omega_P.append([2.0 * p for p in P_half_list])
        all_Omega_CD.append([2.0 * cd for cd in cd_half_list])
        t_list_all.append(tstep)

        H_terms = []
        H_terms.append([stokes_op(phi), make_coeff_from_arr(S_half)])
        for k, op in enumerate(pump_ops):
            H_terms.append([op, make_coeff_from_arr(P_half_list[k])])

        cd_ops = step.get('cd_ops', [])
        for k, op in enumerate(cd_ops):
            if k < len(cd_half_list):
                H_terms.append([op, make_coeff_from_arr(cd_half_list[k])])

        args = {'tlist': tstep}
        result = mesolve(H_terms, psi, tstep, [], args=args, options=solver_opts(T))
        psi = result.states[-1]
    return psi, all_Omega_S, all_Omega_P, all_Omega_CD, t_list_all


## 1. quartit、qubit 的 STIRSAP、STIRAP 扫描（T = 1–300）

### 1a. quartit STIRSAP

In [ ]:
import numpy as np
from qutip import *

import qutip as _qt
_QT5 = int(_qt.__version__.split('.')[0]) >= 5
def solver_opts(T_val):
    od = dict(nsteps=100000, atol=1e-10, rtol=1e-8, max_step=T_val/10)
    return od if _QT5 else Options(**od)

import matplotlib.pyplot as plt
import time
from tqdm import tqdm
from scipy.ndimage import uniform_filter1d

n_states = 6
quartit_dim = 4
smooth_window = 1
N_per_step = 1000

sigma_sg = 3.0
p_sg = 10

a = basis(6, 0)
b = basis(6, 1)
c = basis(6, 2)
d = basis(6, 3)
e = basis(6, 4)
f = basis(6, 5)

F4 = 0.5 * np.array([[1, 1, 1, 1],
                     [1, 1j, -1, -1j],
                     [1, -1, 1, -1],
                     [1, -1j, -1, 1j]])

def smooth_deriv(y, dt, window=smooth_window):
    dy = np.gradient(y, dt)
    return uniform_filter1d(dy, size=window)

def stokes_op(phi):
    return a*b.dag()*np.conj(np.exp(1j*phi)) + b*a.dag()*np.exp(1j*phi)

def compute_cd_half_list(tlist, tau, amps, ampp_list, psi, T):
    dt = tlist[1] - tlist[0]
    S_half = 0.5 * amps * np.exp(-(tlist + tau + psi)**2 / T**2)
    Omega_S = 2.0 * S_half

    P_half_raw_list = [0.5 * amp * np.exp(-(tlist + psi)**2 / T**2) for amp in ampp_list]
    Omega_P_list = [2.0 * p for p in P_half_raw_list]

    Omega_P_sq = sum(p**2 for p in Omega_P_list)
    Omega_P = np.sqrt(Omega_P_sq + 1e-12)

    dOmega_S = smooth_deriv(Omega_S, dt)
    dOmega_P_list = [smooth_deriv(p, dt) for p in Omega_P_list]

    if len(Omega_P_list) == 1:
        dOmega_P = dOmega_P_list[0]
    else:
        dOmega_P = sum(p*dp for p, dp in zip(Omega_P_list, dOmega_P_list)) / (Omega_P + 1e-12)

    theta_dot = (dOmega_P * Omega_S - dOmega_S * Omega_P) / (Omega_P**2 + Omega_S**2 + 1e-9)

    tc = -psi
    super_gaussian_window = np.exp(-(np.abs(tlist - tc) / (sigma_sg * T))**p_sg)

    theta_dot_windowed = theta_dot * super_gaussian_window
    Omega_cd = 2.0 * theta_dot_windowed

    scale = Omega_cd / (Omega_P + 1e-12)
    Omega_cd_list = [p * scale for p in Omega_P_list]
    cd_half_list = [0.5 * val for val in Omega_cd_list]

    return cd_half_list

steps_def = [
    {
        'phi': 0,
        'tau_factor': 1,
        'amps': 2.0,
        'ampp_list': [np.sqrt(2), -np.sqrt(2)],
        'pump_ops': [
            b*d.dag() + d*b.dag(),
            b*f.dag() + f*b.dag()
        ],
        'cd_ops': [
            a*d.dag()*np.conj(np.exp(1j*(0+0.5*np.pi))) + d*a.dag()*np.exp(1j*(0+0.5*np.pi)),
            a*f.dag()*np.conj(np.exp(1j*(0+0.5*np.pi))) + f*a.dag()*np.exp(1j*(0+0.5*np.pi))
        ]
    },
    {
        'phi': np.pi/2,
        'tau_factor': -1,
        'amps': 2.0,
        'ampp_list': [np.sqrt(2), -np.sqrt(2)],
        'pump_ops': [
            b*d.dag() + d*b.dag(),
            b*f.dag() + f*b.dag()
        ],
        'cd_ops': [
            a*d.dag()*np.conj(np.exp(1j*(np.pi/2+0.5*np.pi))) + d*a.dag()*np.exp(1j*(np.pi/2+0.5*np.pi)),
            a*f.dag()*np.conj(np.exp(1j*(np.pi/2+0.5*np.pi))) + f*a.dag()*np.exp(1j*(np.pi/2+0.5*np.pi))
        ]
    },
    {
        'phi': 0,
        'tau_factor': 1,
        'amps': 2.0,
        'ampp_list': [-1.0, 1.0, 1.0, 1.0],
        'pump_ops': [
            b*c.dag() + c*b.dag(),
            b*d.dag() + d*b.dag(),
            b*e.dag() + e*b.dag(),
            b*f.dag() + f*b.dag()
        ],
        'cd_ops': [
            a*c.dag()*np.conj(np.exp(1j*(0+0.5*np.pi))) + c*a.dag()*np.exp(1j*(0+0.5*np.pi)),
            a*d.dag()*np.conj(np.exp(1j*(0+0.5*np.pi))) + d*a.dag()*np.exp(1j*(0+0.5*np.pi)),
            a*e.dag()*np.conj(np.exp(1j*(0+0.5*np.pi))) + e*a.dag()*np.exp(1j*(0+0.5*np.pi)),
            a*f.dag()*np.conj(np.exp(1j*(0+0.5*np.pi))) + f*a.dag()*np.exp(1j*(0+0.5*np.pi))
        ]
    },
    {
        'phi': np.pi,
        'tau_factor': -1,
        'amps': 2.0,
        'ampp_list': [-1.0, 1.0, 1.0, 1.0],
        'pump_ops': [
            b*c.dag() + c*b.dag(),
            b*d.dag() + d*b.dag(),
            b*e.dag() + e*b.dag(),
            b*f.dag() + f*b.dag()
        ],
        'cd_ops': [
            a*c.dag()*np.conj(np.exp(1j*(np.pi+0.5*np.pi))) + c*a.dag()*np.exp(1j*(np.pi+0.5*np.pi)),
            a*d.dag()*np.conj(np.exp(1j*(np.pi+0.5*np.pi))) + d*a.dag()*np.exp(1j*(np.pi+0.5*np.pi)),
            a*e.dag()*np.conj(np.exp(1j*(np.pi+0.5*np.pi))) + e*a.dag()*np.exp(1j*(np.pi+0.5*np.pi)),
            a*f.dag()*np.conj(np.exp(1j*(np.pi+0.5*np.pi))) + f*a.dag()*np.exp(1j*(np.pi+0.5*np.pi))
        ]
    }
]

def make_coeff_from_arr(arr):
    def f(t, args):
        return np.interp(t, args['tlist'], arr)
    return f

def evolve_one_step(psi0, step_idx, T):
    step = steps_def[step_idx]
    phi = step['phi']

    tau = step['tau_factor'] * (T / 0.8)

    amps = step['amps']
    ampp_list = step['ampp_list']
    pump_ops = step['pump_ops']
    cd_ops = step['cd_ops']

    psi_shift = -(2*step_idx + 1) * 5 * T
    t_start = step_idx * 10 * T
    t_end = (step_idx + 1) * 10 * T
    tstep = np.linspace(t_start, t_end, N_per_step)

    S_half = 0.5 * amps * np.exp(-(tstep + tau + psi_shift)**2 / T**2)
    P_half_list = [0.5 * amp * np.exp(-(tstep + psi_shift)**2 / T**2) for amp in ampp_list]

    cd_half_list = compute_cd_half_list(tstep, tau, amps, ampp_list, psi_shift, T)

    H_terms = []
    H_terms.append([stokes_op(phi), make_coeff_from_arr(S_half)])
    for k in range(len(pump_ops)):
        H_terms.append([pump_ops[k], make_coeff_from_arr(P_half_list[k])])
    for k in range(len(cd_ops)):
        H_terms.append([cd_ops[k], make_coeff_from_arr(cd_half_list[k])])

    args = {'tlist': tstep}
    result = mesolve(H_terms, psi0, tstep, [], args=args, options=solver_opts(T))
    return result.states[-1]

def run_full_sequence(psi0, T):
    psi = psi0
    for step_idx in range(4):
        psi = evolve_one_step(psi, step_idx, T)
    return psi

def compute_gate_fidelity(T):
    comp_basis = [c, d, e, f]
    final_states = []
    for ket in comp_basis:
        final_ket = run_full_sequence(ket, T)
        final_states.append(final_ket)

    U_impl = np.zeros((4,4), dtype=complex)
    for j, fket in enumerate(final_states):
        for i, bra_ket in enumerate(comp_basis):
            U_impl[i,j] = bra_ket.dag() * fket

    M = F4.conj().T @ U_impl
    F_avg = (np.trace(M @ M.conj().T).real + np.abs(np.trace(M))**2) / (quartit_dim * (quartit_dim + 1))

    leakage_total = 0.0
    for j in range(4):
        pop_in_comp = np.sum(np.abs(U_impl[:, j])**2)
        leakage_total += (1.0 - pop_in_comp)
    avg_leakage = leakage_total / 4.0

    unitary_err = np.max(np.abs(U_impl.conj().T @ U_impl - np.eye(4)))

    return F_avg, avg_leakage, unitary_err, U_impl

T_values = np.arange(1, 301, 0.5)
n_T = len(T_values)

fidelities = np.zeros(n_T)
avg_leakages = np.zeros(n_T)
unitary_errors = np.zeros(n_T)

print(f"扫描 {n_T} 个 T 值...")
start_time = time.time()

for idx, T in enumerate(tqdm(T_values, desc="扫描 quartit T")):
    F_avg, leak, uerr, _ = compute_gate_fidelity(T)
    fidelities[idx] = F_avg
    avg_leakages[idx] = leak
    unitary_errors[idx] = uerr

total_time = time.time() - start_time
print(f"扫描完成，总用时 {total_time:.2f} 秒")

threshold_099 = 0.99
threshold_0999 = 0.999
first_099 = None
first_0999 = None
for i, fval in enumerate(fidelities):
    if fval >= threshold_099 and first_099 is None:
        first_099 = T_values[i]
    if fval >= threshold_0999 and first_0999 is None:
        first_0999 = T_values[i]

print(f"首次达到 0.99: T = {first_099}")
print(f"首次达到 0.999: T = {first_0999}")

data = np.column_stack((T_values, fidelities, avg_leakages, unitary_errors))
header = "T,AGF,Avg_Leakage,Unitary_Error"
np.savetxt('quartit_numCD_SG_window_results.csv', data, delimiter=',', header=header, comments='')
print("数据已保存为 quartit_numCD_SG_window_results.csv")

plt.figure(figsize=(10,6))
plt.plot(T_values, fidelities, 'b-', linewidth=2)
plt.axhline(0.99, color='r', linestyle='--', alpha=0.7)
plt.axhline(0.999, color='g', linestyle='--', alpha=0.7)
plt.xlabel('T')
plt.ylabel('Average Gate Fidelity')
plt.title('Quartit QFT Explicit CD (Super-Gaussian Window + tau=T/0.8)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('quartit_numCD_SG_window_AGF_vs_T.png', dpi=200)
plt.show()


### 1b. qubit STIRAP + qubit STIRSAP

In [ ]:
basis_states = [basis(6, 2), basis(6, 3), basis(6, 4), basis(6, 5)]

def scan_protocol(name, steps, waveform, T_values):
    n_T = len(T_values)
    fidelities = np.zeros(n_T)
    leakages = np.zeros(n_T)
    for idx, T in enumerate(tqdm(T_values, desc=name)):
        final_states = [run_protocol(k0, T, steps, waveform)[0] for k0 in basis_states]
        U_impl = np.zeros((4, 4), dtype=complex)
        for j, fket in enumerate(final_states):
            for i, bk in enumerate(basis_states):
                U_impl[i, j] = bk.dag() * fket
        M = F4.conj().T @ U_impl
        fidelities[idx] = (np.trace(M @ M.conj().T).real + np.abs(np.trace(M))**2) / 20.0
        leakages[idx] = 1.0 - np.trace(U_impl @ U_impl.conj().T).real / 4.0
    np.savetxt(name + '_T08_AGF_scan.csv',
               np.column_stack((T_values, fidelities, leakages)),
               delimiter=',', header='T,AGF,Avg_Leakage', comments='')
    return fidelities

T_scan = np.arange(1, 301, 0.5)
plt.figure(figsize=(10, 6))
for name, steps, wf in [('qubit_STIRAP', qubit_STIRAP_steps, get_STIRAP_waveforms),
                        ('qubit_STIRSAP', qubit_STIRSAP_steps, get_STIRSAP_waveforms)]:
    fids = scan_protocol(name, steps, wf, T_scan)
    plt.plot(T_scan, fids, label=name)
plt.axhline(0.99, color='r', linestyle='--', alpha=0.7)
plt.axhline(0.999, color='g', linestyle='--', alpha=0.7)
plt.xlabel('T')
plt.ylabel('Average Gate Fidelity')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('qubit_STIRAP_STIRSAP_AGF_vs_T.png', dpi=200)
plt.show()


### 1c. quartit STIRAP 见第 6 节

## 2. 四个协议初态敏感性

In [ ]:
sens_protocols = {
    'qubit_STIRAP':    (qubit_STIRAP_steps,    get_STIRAP_waveforms),
    'qubit_STIRSAP':   (qubit_STIRSAP_steps,   get_STIRSAP_waveforms),
    'quartit_STIRAP':  (quartit_STIRAP_steps,  get_STIRAP_waveforms),
    'quartit_STIRSAP': (quartit_STIRSAP_steps, get_STIRSAP_waveforms),
}

num_samples = 200
np.random.seed()
rand_kets_4d = [rand_ket(4) for _ in range(num_samples)]
comp_kets = [basis(6, 2), basis(6, 3), basis(6, 4), basis(6, 5)]

init_states = []
ideal_states = []
for ket in rand_kets_4d:
    v = ket.full().flatten()
    init_states.append(sum(v[i] * comp_kets[i] for i in range(4)))
    w = F4 @ v
    ideal_states.append(sum(w[i] * comp_kets[i] for i in range(4)))

T_sens = np.linspace(0.1, 20, 5)
for name, (steps, waveform) in sens_protocols.items():
    avg_f = np.zeros(len(T_sens))
    max_f = np.zeros(len(T_sens))
    min_f = np.zeros(len(T_sens))
    for idx, T in enumerate(tqdm(T_sens, desc=name)):
        fids = []
        for k in range(num_samples):
            psi_final = run_protocol(init_states[k], T, steps, waveform)[0]
            fids.append(float(np.abs(ideal_states[k].dag() * psi_final)**2))
        fids = np.array(fids)
        avg_f[idx] = fids.mean()
        max_f[idx] = fids.max()
        min_f[idx] = fids.min()
    np.savetxt(name + '_state_fidelity.csv',
               np.column_stack((T_sens, avg_f, max_f, min_f)),
               delimiter=',', header='T,Avg_Fid,Max_Fid,Min_Fid', comments='')
    plt.figure(figsize=(8, 5))
    plt.plot(T_sens, avg_f, 'b-', label='Average fidelity')
    plt.fill_between(T_sens, min_f, max_f, alpha=0.2, color='b', label='Min-Max range')
    plt.xlabel('T')
    plt.ylabel('State fidelity')
    plt.title(name)
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(name + '_state_fidelity.png', dpi=150)
    plt.show()


## 3. 初态依赖热图

In [ ]:
import numpy as np
from qutip import *
import matplotlib.pyplot as plt
from tqdm import tqdm

import qutip as _qt
_QT5 = int(_qt.__version__.split('.')[0]) >= 5
def solver_opts(T_val):
    od = dict(nsteps=100000, atol=1e-10, rtol=1e-8, max_step=T_val/10)
    return od if _QT5 else Options(**od)

T_val = 3
N_per_step = 1000

sigma_sg = 3.0
p_sg = 10

a = basis(6, 0)
b = basis(6, 1)
c = basis(6, 2)
d = basis(6, 3)
e = basis(6, 4)
f = basis(6, 5)

F4 = 0.5 * np.array([[1, 1, 1, 1],
                     [1, 1j, -1, -1j],
                     [1, -1, 1, -1],
                     [1, -1j, -1, 1j]])

def stokes_op(phi):
    return a * b.dag() * np.conj(np.exp(1j*phi)) + b * a.dag() * np.exp(1j*phi)

def make_coeff_from_arr(arr):
    def f(t, args):
        return np.interp(t, args['tlist'], arr)
    return f

steps_def = [
    {'phi': 0, 'tau_factor': 1, 'amps': 2.0,
     'ampp_list': [np.sqrt(2), -np.sqrt(2)],
     'pump_ops': [b*d.dag() + d*b.dag(), b*f.dag() + f*b.dag()]},
    {'phi': np.pi/2, 'tau_factor': -1, 'amps': 2.0,
     'ampp_list': [np.sqrt(2), -np.sqrt(2)],
     'pump_ops': [b*d.dag() + d*b.dag(), b*f.dag() + f*b.dag()]},
    {'phi': 0, 'tau_factor': 1, 'amps': 2.0,
     'ampp_list': [-1.0, 1.0, 1.0, 1.0],
     'pump_ops': [b*c.dag() + c*b.dag(), b*d.dag() + d*b.dag(),
                  b*e.dag() + e*b.dag(), b*f.dag() + f*b.dag()]},
    {'phi': np.pi, 'tau_factor': -1, 'amps': 2.0,
     'ampp_list': [-1.0, 1.0, 1.0, 1.0],
     'pump_ops': [b*c.dag() + c*b.dag(), b*d.dag() + d*b.dag(),
                  b*e.dag() + e*b.dag(), b*f.dag() + f*b.dag()]}
]

def evolve_one_step(psi0, step_idx, T):
    step = steps_def[step_idx]
    phi = step['phi']

    tau = step['tau_factor'] * (T / 0.8)
    amps = step['amps']
    ampp_list = step['ampp_list']
    pump_ops = step['pump_ops']

    psi_shift = -(2*step_idx + 1) * 5 * T
    t_start = step_idx * 10 * T
    t_end = (step_idx + 1) * 10 * T
    tstep = np.linspace(t_start, t_end, N_per_step)

    S_half_raw = 0.5 * amps * np.exp(-(tstep + tau + psi_shift)**2 / T**2)
    P_half_raw = [0.5 * amp * np.exp(-(tstep + psi_shift)**2 / T**2) for amp in ampp_list]

    tc = -psi_shift
    window = np.exp(-(np.abs(tstep - tc) / (sigma_sg * T))**p_sg)
    S_half = S_half_raw * window
    P_half = [p * window for p in P_half_raw]

    H_terms = [[stokes_op(phi), make_coeff_from_arr(S_half)]]
    for k, op in enumerate(pump_ops):
        H_terms.append([op, make_coeff_from_arr(P_half[k])])

    args = {'tlist': tstep}
    result = mesolve(H_terms, psi0, tstep, [], args=args, options=solver_opts(T))
    return result.states[-1]

def run_full_sequence(psi0, T):
    psi = psi0
    for step_idx in range(4):
        psi = evolve_one_step(psi, step_idx, T)
    return psi

comp_basis = [c, d, e, f]
final_states = [run_full_sequence(ket, T_val) for ket in comp_basis]

U_impl = np.zeros((4, 4), dtype=complex)
for j, fket in enumerate(final_states):
    for i, bra_ket in enumerate(comp_basis):
        overlap = bra_ket.dag() * fket
        if isinstance(overlap, Qobj):
            overlap = overlap.full()[0, 0]
        U_impl[i, j] = overlap

print("Implemented U matrix (columns: initial |0>,|1>,|2>,|3>):")
print(np.round(U_impl, 4))

print("\nOverlap of each column with ideal QFT column:")
for j in range(4):
    ideal_col = F4[:, j]
    impl_col = U_impl[:, j]
    overlap = np.vdot(ideal_col, impl_col)
    print(f" Column {j}: overlap magnitude = {np.abs(overlap):.6f}, phase = {np.angle(overlap):.4f} rad")

alpha_vals = np.linspace(0, np.pi/2, 49)
delta_vals = np.linspace(0, 2*np.pi, 49)

fid_grid = np.zeros((len(alpha_vals), len(delta_vals)))
for i, alpha in enumerate(tqdm(alpha_vals, desc="Scanning superposition")):
    for j, delta in enumerate(delta_vals):

        psi_init = np.cos(alpha)*d + np.sin(alpha)*np.exp(1j*delta)*f
        psi_final = run_full_sequence(psi_init, T_val)

        target_vec = np.cos(alpha)*F4[:,1] + np.sin(alpha)*np.exp(1j*delta)*F4[:,3]
        psi_target = target_vec[0]*c + target_vec[1]*d + target_vec[2]*e + target_vec[3]*f
        fid = np.abs(psi_target.dag() * psi_final)**2
        fid_grid[i, j] = fid

min_idx = np.unravel_index(np.argmin(fid_grid), fid_grid.shape)
alpha_min = alpha_vals[min_idx[0]]
delta_min = delta_vals[min_idx[1]]
fid_min = fid_grid[min_idx]
print(f"\nMinimum fidelity = {fid_min:.6f}")
print(f"At alpha = {alpha_min:.4f} rad (weight ratio |3>/|1> = {np.tan(alpha_min):.4f})")
print(f"Relative phase delta = {delta_min:.4f} rad")

fids_vs_delta = fid_grid[min_idx[0], :]
plt.figure(figsize=(10,5))
plt.plot(delta_vals, fids_vs_delta, 'b-')
plt.axhline(y=fid_min, color='r', linestyle='--', label=f'Min = {fid_min:.4f}')
plt.axhline(y=0.5479, color='gray', linestyle='--', label='Pure |3> fid (reference)')
plt.xlabel(r'Relative phase $\delta$ (rad)')
plt.ylabel('Fidelity')
plt.title(f'Fidelity vs. relative phase (alpha = {alpha_min:.3f} rad)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('superposition_scan_phase.png', dpi=150)
plt.show()


In [ ]:
alpha_grid, delta_grid = np.meshgrid(alpha_vals, delta_vals, indexing='ij')

alpha_flat = alpha_grid.ravel()
delta_flat = delta_grid.ravel()
fid_flat = fid_grid.ravel()

csv_data = np.column_stack((alpha_flat, delta_flat, fid_flat))
header = "alpha,delta,fidelity"
np.savetxt("superposition_scan_results.csv", csv_data, delimiter=',', header=header, comments='')
print("数据已保存至 superposition_scan_results.csv")


## 4. all-optical 的 AGF 扫描（T = 1–300）

### 4a. qubit all-optical

In [ ]:
import numpy as np
if not hasattr(np, 'trapz'):
    np.trapz = np.trapezoid
from qutip import *

import qutip as _qt
_QT5 = int(_qt.__version__.split('.')[0]) >= 5
def solver_opts(T_val):
    od = dict(nsteps=100000, atol=1e-10, rtol=1e-8, max_step=T_val/10)
    return od if _QT5 else Options(**od)

import matplotlib.pyplot as plt
import time
from tqdm import tqdm

np.seterr(divide='ignore', invalid='ignore')

n_states = 6
qubit_dim = 4
N_per_step = 1000

a = basis(6, 0)
b = basis(6, 1)
c = basis(6, 2)
d = basis(6, 3)
e = basis(6, 4)
f = basis(6, 5)

F4 = 0.5 * np.array([[1, 1, 1, 1],
                     [1, 1j, -1, -1j],
                     [1, -1, 1, -1],
                     [1, -1j, -1, 1j]])

def stokes_op(phi):
    return a * b.dag() * np.conj(np.exp(1j*phi)) + b * a.dag() * np.exp(1j*phi)

def get_all_optical_waveforms(tlist, tau, amps, ampp_list, psi, T, eps=1e-12):
    dt = tlist[1] - tlist[0]

    S_half_raw = 0.5 * amps * np.exp(-(tlist + tau + psi)**2 / T**2)
    P_half_raw_list = [0.5 * amp * np.exp(-(tlist + psi)**2 / T**2) for amp in ampp_list]

    Omega_S = 2.0 * S_half_raw
    Omega_P_list = [2.0 * p for p in P_half_raw_list]
    Omega_P = np.sqrt(sum(p**2 for p in Omega_P_list))

    dOmega_P = np.gradient(Omega_P, dt)
    dOmega_S = np.gradient(Omega_S, dt)

    denom = Omega_P**2 + Omega_S**2
    theta_dot = np.divide(dOmega_P * Omega_S - dOmega_S * Omega_P,
                          denom,
                          where=(denom >= eps),
                          out=np.zeros_like(Omega_S))

    sigma_sg = 3.0
    p_sg = 10
    tc = -psi
    super_gaussian_window = np.exp(-(np.abs(tlist - tc) / (sigma_sg * T))**p_sg)
    Omega_a = 2.0 * theta_dot * super_gaussian_window

    phi_rot = np.arctan2(Omega_a, Omega_P)
    dphi_rot = np.gradient(phi_rot, dt)

    Omega_P_tilde = np.sqrt(Omega_P**2 + Omega_a**2)
    Omega_S_tilde = Omega_S - 2.0 * dphi_rot

    scale = np.ones_like(Omega_P)
    mask_P = Omega_P > eps
    scale[mask_P] = Omega_P_tilde[mask_P] / Omega_P[mask_P]

    S_half_mod = Omega_S_tilde / 2.0
    P_half_mod_list = [0.5 * p * scale for p in Omega_P_list]
    return S_half_mod, P_half_mod_list

def make_coeff_from_arr(arr):
    def f(t, args):
        return np.interp(t, args['tlist'], arr)
    return f

steps_def = [

    {'phi': 0, 'tau_factor': 1, 'amps': 2.0,
     'ampp_list': [np.sqrt(2 - np.sqrt(2)), np.sqrt(2 + np.sqrt(2))],
     'pump_ops': [-b*d.dag()-d*b.dag(), b*f.dag()+f*b.dag()]},

    {'phi': np.pi, 'tau_factor': -1, 'amps': 2.0,
     'ampp_list': [np.sqrt(2 - np.sqrt(2)), np.sqrt(2 + np.sqrt(2))],
     'pump_ops': [-b*d.dag()-d*b.dag(), b*f.dag()+f*b.dag()]},

    {'phi': 0, 'tau_factor': 1, 'amps': 2.0,
     'ampp_list': [np.sqrt(2 - np.sqrt(2)), np.sqrt(2 + np.sqrt(2))],
     'pump_ops': [-b*c.dag()-c*b.dag(), b*e.dag()+e*b.dag()]},

    {'phi': np.pi, 'tau_factor': -1, 'amps': 2.0,
     'ampp_list': [np.sqrt(2 - np.sqrt(2)), np.sqrt(2 + np.sqrt(2))],
     'pump_ops': [-b*c.dag()-c*b.dag(), b*e.dag()+e*b.dag()]},

    {'phi': 0, 'tau_factor': 1, 'amps': 2.0,
     'ampp_list': [2.0],
     'pump_ops': [b*f.dag()+f*b.dag()]},

    {'phi': 0.5*np.pi, 'tau_factor': -1, 'amps': 2.0,
     'ampp_list': [2.0],
     'pump_ops': [b*f.dag()+f*b.dag()]},

    {'phi': 0, 'tau_factor': 1, 'amps': 2.0,
     'ampp_list': [np.sqrt(2 - np.sqrt(2)), np.sqrt(2 + np.sqrt(2))],
     'pump_ops': [b*e.dag()+e*b.dag(), -b*f.dag()-f*b.dag()]},

    {'phi': np.pi, 'tau_factor': -1, 'amps': 2.0,
     'ampp_list': [np.sqrt(2 - np.sqrt(2)), np.sqrt(2 + np.sqrt(2))],
     'pump_ops': [b*e.dag()+e*b.dag(), -b*f.dag()-f*b.dag()]},

    {'phi': 0, 'tau_factor': 1, 'amps': 2.0,
     'ampp_list': [np.sqrt(2 - np.sqrt(2)), np.sqrt(2 + np.sqrt(2))],
     'pump_ops': [b*c.dag()+c*b.dag(), -b*d.dag()-d*b.dag()]},

    {'phi': np.pi, 'tau_factor': -1, 'amps': 2.0,
     'ampp_list': [np.sqrt(2 - np.sqrt(2)), np.sqrt(2 + np.sqrt(2))],
     'pump_ops': [b*c.dag()+c*b.dag(), -b*d.dag()-d*b.dag()]},

    {'phi': 0, 'tau_factor': 1, 'amps': 2.0,
     'ampp_list': [np.sqrt(2), np.sqrt(2)],
     'pump_ops': [-b*d.dag()-d*b.dag(), b*e.dag()+e*b.dag()]},

    {'phi': np.pi, 'tau_factor': -1, 'amps': 2.0,
     'ampp_list': [np.sqrt(2), np.sqrt(2)],
     'pump_ops': [-b*d.dag()-d*b.dag(), b*e.dag()+e*b.dag()]},
]

def evolve_one_step(psi0, step_idx, T):
    step = steps_def[step_idx]
    phi = step['phi']
    tau = step['tau_factor'] * (T / 0.8)
    amps = step['amps']
    ampp_list = step['ampp_list']
    pump_ops = step['pump_ops']

    psi_shift = -(2*step_idx + 1) * 5 * T
    t_start = step_idx * 10 * T
    t_end = (step_idx + 1) * 10 * T
    tstep = np.linspace(t_start, t_end, N_per_step)

    S_mod, P_mod_list = get_all_optical_waveforms(tstep, tau, amps, ampp_list, psi_shift, T)

    H_terms = []
    H_terms.append([stokes_op(phi), make_coeff_from_arr(S_mod)])
    for k, op in enumerate(pump_ops):
        H_terms.append([op, make_coeff_from_arr(P_mod_list[k])])

    args = {'tlist': tstep}
    result = mesolve(H_terms, psi0, tstep, [], args=args, options=solver_opts(T))
    return result.states[-1]

def run_full_sequence(psi0, T):
    psi = psi0
    for step_idx in range(12):
        psi = evolve_one_step(psi, step_idx, T)
    return psi

def compute_gate_fidelity(T):
    comp_basis = [c, d, e, f]
    final_states = [run_full_sequence(ket, T) for ket in comp_basis]

    U_impl = np.zeros((4,4), dtype=complex)
    for j, fket in enumerate(final_states):
        for i, bra_ket in enumerate(comp_basis):
            U_impl[i,j] = bra_ket.dag() * fket

    M = F4.conj().T @ U_impl
    F_avg = (np.trace(M @ M.conj().T).real + np.abs(np.trace(M))**2) / (qubit_dim * (qubit_dim + 1))

    leakage_total = 0.0
    for j in range(4):
        pop_in_comp = np.sum(np.abs(U_impl[:, j])**2)
        leakage_total += (1.0 - pop_in_comp)
    avg_leakage = leakage_total / 4.0

    unitary_err = np.max(np.abs(U_impl.conj().T @ U_impl - np.eye(4)))

    total_area = 0.0
    omega_peak = 0.0
    for step_idx in range(12):
        step = steps_def[step_idx]
        tau = step['tau_factor'] * (T / 0.8)
        psi_shift = -(2*step_idx + 1) * 5 * T
        t_start = step_idx * 10 * T
        t_end = (step_idx + 1) * 10 * T
        tstep = np.linspace(t_start, t_end, N_per_step)
        S_mod, P_mod_list = get_all_optical_waveforms(tstep, tau, step['amps'],
                                                      step['ampp_list'], psi_shift, T)
        Omega_S = 2.0 * S_mod
        Omega_P_list = [2.0 * p for p in P_mod_list]
        peak_step = max(np.max(np.abs(Omega_S)),
                        max(np.max(np.abs(op)) for op in Omega_P_list))
        omega_peak = max(omega_peak, peak_step)
        integrand = Omega_S**2 + sum(op**2 for op in Omega_P_list)
        total_area += np.trapz(integrand, tstep)

    return F_avg, avg_leakage, unitary_err, U_impl, omega_peak, total_area

T_values = np.arange(1, 301, 0.5)
n_T = len(T_values)

fidelities = np.zeros(n_T)
avg_leakages = np.zeros(n_T)
unitary_errors = np.zeros(n_T)
omega_peaks = np.zeros(n_T)
total_areas = np.zeros(n_T)

print(f"扫描 {n_T} 个 T 值 (qubit all-optical, T/0.8 + SG window)...")
start_time = time.time()

for idx, T in enumerate(tqdm(T_values, desc="扫描 T")):
    F_avg, leak, uerr, _, peak, area = compute_gate_fidelity(T)
    fidelities[idx] = F_avg
    avg_leakages[idx] = leak
    unitary_errors[idx] = uerr
    omega_peaks[idx] = peak
    total_areas[idx] = area

total_time = time.time() - start_time
print(f"扫描完成，总用时 {total_time:.2f} 秒")

first_099 = first_0999 = None
for i, fval in enumerate(fidelities):
    if fval >= 0.99 and first_099 is None:
        first_099 = T_values[i]
    if fval >= 0.999 and first_0999 is None:
        first_0999 = T_values[i]

print(f"首次达到 0.99: T = {first_099}")
print(f"首次达到 0.999: T = {first_0999}")
print(f"最高保真度: {fidelities.max():.8f} at T = {T_values[fidelities.argmax()]}")

np.savetxt('qubit_alloptical_T08_SGwindow_energy.csv',
           np.column_stack((T_values, fidelities, omega_peaks, total_areas)),
           delimiter=',', header='T,AGF,Omega_peak,Total_Area', comments='')
np.savetxt('qubit_alloptical_T08_SGwindow_details.csv',
           np.column_stack((T_values, fidelities, avg_leakages, unitary_errors)),
           delimiter=',', header='T,AGF,Avg_Leakage,Unitary_Error', comments='',
           fmt='%.1f,%.9f,%.6f,%.6f')

plt.figure(figsize=(10,6))
plt.plot(T_values, fidelities, 'orange', linewidth=2)
plt.axhline(0.99, color='gray', linestyle='--')
plt.axhline(0.999, color='gray', linestyle='-.')
plt.xlabel('T')
plt.ylabel('Average Gate Fidelity')
plt.title('Qubit QFT via All-optical STIRSAP (T/0.8 + SG window)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('qubit_alloptical_T08_SGwindow_AGF.png', dpi=200)
plt.show()


### 4b. quartit all-optical

In [ ]:
import numpy as np
from qutip import *

import qutip as _qt
_QT5 = int(_qt.__version__.split('.')[0]) >= 5
def solver_opts(T_val):
    od = dict(nsteps=100000, atol=1e-10, rtol=1e-8, max_step=T_val/10)
    return od if _QT5 else Options(**od)

import matplotlib.pyplot as plt
import time
from tqdm import tqdm

np.seterr(divide='ignore', invalid='ignore')

n_states = 6
quartit_dim = 4
N_per_step = 1000

sigma_sg = 3.0
p_sg = 10

a = basis(6, 0)
b = basis(6, 1)
c = basis(6, 2)
d = basis(6, 3)
e = basis(6, 4)
f = basis(6, 5)

F4 = 0.5 * np.array([[1, 1, 1, 1],
                     [1, 1j, -1, -1j],
                     [1, -1, 1, -1],
                     [1, -1j, -1, 1j]])

def get_all_optical_waveforms(tlist, tau, amps, ampp_list, psi, T):
    dt = tlist[1] - tlist[0]

    S_half_raw = 0.5 * amps * np.exp(-(tlist + tau + psi)**2 / T**2)
    P_half_raw_list = [0.5 * amp * np.exp(-(tlist + psi)**2 / T**2) for amp in ampp_list]

    Omega_S = 2.0 * S_half_raw
    Omega_P_list = [2.0 * p for p in P_half_raw_list]
    Omega_P = np.sqrt(sum(p**2 for p in Omega_P_list))

    dOmega_P = np.gradient(Omega_P, dt)
    dOmega_S = np.gradient(Omega_S, dt)

    denom = Omega_P**2 + Omega_S**2
    theta_dot = np.divide(dOmega_P * Omega_S - dOmega_S * Omega_P,
                          denom,
                          where=(denom >= 1e-250),
                          out=np.zeros_like(Omega_S))

    tc = -psi
    super_gaussian_window = np.exp(-(np.abs(tlist - tc) / (sigma_sg * T))**p_sg)

    Omega_a = 2.0 * theta_dot * super_gaussian_window

    phi_rot = np.arctan2(Omega_a, Omega_P)
    dphi_rot = np.gradient(phi_rot, dt)

    Omega_P_tilde = np.sqrt(Omega_P**2 + Omega_a**2)
    Omega_S_tilde = Omega_S - 2.0 * dphi_rot

    scale = np.divide(Omega_P_tilde, Omega_P, where=(Omega_P > 1e-100), out=np.ones_like(Omega_P))

    S_half_mod = Omega_S_tilde / 2.0
    P_half_mod_list = [0.5 * p * scale for p in Omega_P_list]

    return S_half_mod, P_half_mod_list

def stokes_op(phase):
    return a * b.dag() * np.conj(np.exp(1j*phase)) + b * a.dag() * np.exp(1j*phase)

steps_def = [
    {'stokes_phase': 0, 'tau_factor': 1, 'amps': 2.0,
     'ampp_list': [np.sqrt(2), -np.sqrt(2)],
     'pump_ops': [b*d.dag()+d*b.dag(), b*f.dag()+f*b.dag()]},
    {'stokes_phase': np.pi/2, 'tau_factor': -1, 'amps': 2.0,
     'ampp_list': [np.sqrt(2), -np.sqrt(2)],
     'pump_ops': [b*d.dag()+d*b.dag(), b*f.dag()+f*b.dag()]},
    {'stokes_phase': 0, 'tau_factor': 1, 'amps': 2.0,
     'ampp_list': [-1.0, 1.0, 1.0, 1.0],
     'pump_ops': [b*c.dag()+c*b.dag(), b*d.dag()+d*b.dag(),
                  b*e.dag()+e*b.dag(), b*f.dag()+f*b.dag()]},
    {'stokes_phase': np.pi, 'tau_factor': -1, 'amps': 2.0,
     'ampp_list': [-1.0, 1.0, 1.0, 1.0],
     'pump_ops': [b*c.dag()+c*b.dag(), b*d.dag()+d*b.dag(),
                  b*e.dag()+e*b.dag(), b*f.dag()+f*b.dag()]}
]

def make_coeff_from_arr(arr):
    def f(t, args):
        return np.interp(t, args['tlist'], arr)
    return f

def evolve_one_step(psi0, step_idx, T):
    step = steps_def[step_idx]
    stokes_phase = step['stokes_phase']

    tau = step['tau_factor'] * (T / 0.8)

    amps = step['amps']
    ampp_list = step['ampp_list']
    pump_ops = step['pump_ops']

    psi_shift = -(2*step_idx + 1) * 5 * T
    t_start = step_idx * 10 * T
    t_end = (step_idx + 1) * 10 * T
    tstep = np.linspace(t_start, t_end, N_per_step)

    S_mod, P_mod_list = get_all_optical_waveforms(tstep, tau, amps, ampp_list, psi_shift, T)

    H_terms = []
    H_terms.append([stokes_op(stokes_phase), make_coeff_from_arr(S_mod)])
    for k, op in enumerate(pump_ops):
        H_terms.append([op, make_coeff_from_arr(P_mod_list[k])])

    args = {'tlist': tstep}
    result = mesolve(H_terms, psi0, tstep, [], args=args, options=solver_opts(T))
    return result.states[-1]

def run_full_sequence(psi0, T):
    psi = psi0
    for step_idx in range(4):
        psi = evolve_one_step(psi, step_idx, T)
    return psi

def compute_gate_fidelity(T):
    comp_basis = [c, d, e, f]
    final_states = []
    for ket in comp_basis:
        final_ket = run_full_sequence(ket, T)
        final_states.append(final_ket)

    U_impl = np.zeros((4,4), dtype=complex)
    for j, fket in enumerate(final_states):
        for i, bra_ket in enumerate(comp_basis):
            U_impl[i,j] = bra_ket.dag() * fket

    M = F4.conj().T @ U_impl
    F_avg = (np.trace(M @ M.conj().T).real + np.abs(np.trace(M))**2) / (quartit_dim * (quartit_dim + 1))

    leakage_total = 0.0
    for j in range(4):
        pop_in_comp = np.sum(np.abs(U_impl[:, j])**2)
        leakage_total += (1.0 - pop_in_comp)
    avg_leakage = leakage_total / 4.0

    unitary_err = np.max(np.abs(U_impl.conj().T @ U_impl - np.eye(4)))

    return F_avg, avg_leakage, unitary_err

T_values = np.arange(1, 301, 0.5)
n_T = len(T_values)

fidelities = np.zeros(n_T)
avg_leakages = np.zeros(n_T)
unitary_errors = np.zeros(n_T)

print(f"开始扫描 {n_T} 个 T 值，配置: tau=T/0.8, 超高斯平滑窗(sigma=3.0, p=10)...")
start_time = time.time()

for idx, T in enumerate(tqdm(T_values, desc="扫描 quartit all-optical T")):
    F_avg, leak, uerr = compute_gate_fidelity(T)
    fidelities[idx] = F_avg
    avg_leakages[idx] = leak
    unitary_errors[idx] = uerr

total_time = time.time() - start_time
print(f"扫描完成，总用时 {total_time:.2f} 秒")

threshold_099 = 0.99
threshold_0999 = 0.999
first_099 = None
first_0999 = None
for i, fval in enumerate(fidelities):
    if fval >= threshold_099 and first_099 is None:
        first_099 = T_values[i]
    if fval >= threshold_0999 and first_0999 is None:
        first_0999 = T_values[i]

print(f"首次达到 0.99: T = {first_099}")
print(f"首次达到 0.999: T = {first_0999}")

data = np.column_stack((T_values, fidelities, avg_leakages, unitary_errors))
header = "T,AGF,Avg_Leakage,Unitary_Error"
np.savetxt('quartit_alloptical_SG_window_results.csv', data, delimiter=',', header=header, comments='')
print("数据已保存为 quartit_alloptical_SG_window_results.csv")

plt.figure(figsize=(10,6))
plt.plot(T_values, fidelities, 'b-', linewidth=2)
plt.axhline(0.99, color='r', linestyle='--', alpha=0.7)
plt.axhline(0.999, color='g', linestyle='--', alpha=0.7)
plt.xlabel('T')
plt.ylabel('Average Gate Fidelity')
plt.title('Quartit QFT All-optical STIRSAP (Super-Gaussian Window + tau=T/0.8)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('quartit_alloptical_SG_window_AGF_vs_T.png', dpi=200)
plt.show()


## 5. all-optical waveform

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

T = 8.0
step_idx = 0
N_per_step = 1000

phi = 0
tau_factor = 1
amps = 2.0
ampp_list = [np.sqrt(2), -np.sqrt(2)]

tau = tau_factor * (T / 0.8)
psi_shift = -(2 * step_idx + 1) * 5 * T
t_start = step_idx * 10 * T
t_end = (step_idx + 1) * 10 * T
tstep = np.linspace(t_start, t_end, N_per_step)
dt = tstep[1] - tstep[0]

S_half_raw = 0.5 * amps * np.exp(-(tstep + tau + psi_shift)**2 / T**2)
P_half_raw_list = [0.5 * amp * np.exp(-(tstep + psi_shift)**2 / T**2) for amp in ampp_list]

Omega_S = 2.0 * S_half_raw
Omega_P_list = [2.0 * p for p in P_half_raw_list]
Omega_P = np.sqrt(sum(p**2 for p in Omega_P_list))

dOmega_P = np.gradient(Omega_P, dt)
dOmega_S = np.gradient(Omega_S, dt)

eps = 1e-12
denom = Omega_P**2 + Omega_S**2
theta_dot = np.divide(dOmega_P * Omega_S - dOmega_S * Omega_P,
                      denom,
                      where=(denom >= eps),
                      out=np.zeros_like(Omega_S))

sigma_sg = 3.0
p_sg = 10
tc = -psi_shift
window = np.exp(-(np.abs(tstep - tc) / (sigma_sg * T))**p_sg)
Omega_a = 2.0 * theta_dot * window

phi_rot = np.arctan2(Omega_a, Omega_P)
dphi_rot = np.gradient(phi_rot, dt)

Omega_P_tilde = np.sqrt(Omega_P**2 + Omega_a**2)
Omega_S_tilde = Omega_S - 2.0 * dphi_rot

scale = np.ones_like(Omega_P)
mask_P = Omega_P > eps
scale[mask_P] = Omega_P_tilde[mask_P] / Omega_P[mask_P]

S_half_mod = Omega_S_tilde / 2.0
P_half_mod_list = [0.5 * p * scale for p in Omega_P_list]

plt.figure(figsize=(10, 5))
plt.plot(tstep, S_half_mod, 'r-', label='Stokes (half amp)')
for i, p_half in enumerate(P_half_mod_list):
    plt.plot(tstep, p_half, '--', label=f'Pump {i+1} (half amp, amp sign {"+" if ampp_list[i]>0 else "-"})')
plt.xlabel('Time')
plt.ylabel('Half amplitude')
plt.title(f'Quartit all‑optical, step 1, T={T}')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('quartit_alloptical_T1_step1_waveforms.png', dpi=200)
plt.show()

data = np.column_stack((tstep, S_half_mod, P_half_mod_list[0], P_half_mod_list[1]))
header = 't, Stokes_half_amp, Pump1_half_amp, Pump2_half_amp'
np.savetxt('quartit_alloptical_T1_step1_waveforms.csv', data, delimiter=',', header=header, comments='')
print("波形数据已保存为 quartit_alloptical_T1_step1_waveforms.csv")


## 6. Quartit STIRAP

In [ ]:
import numpy as np
from qutip import *

import qutip as _qt
_QT5 = int(_qt.__version__.split('.')[0]) >= 5
def solver_opts(T_val):
    od = dict(nsteps=100000, atol=1e-10, rtol=1e-8, max_step=T_val/10)
    return od if _QT5 else Options(**od)

import matplotlib.pyplot as plt
import time
from tqdm import tqdm

n_states = 6
quartit_dim = 4
N_per_step = 1000

a = basis(6, 0)
b = basis(6, 1)
c = basis(6, 2)
d = basis(6, 3)
e = basis(6, 4)
f = basis(6, 5)

F4 = 0.5 * np.array([[1, 1, 1, 1],
                     [1, 1j, -1, -1j],
                     [1, -1, 1, -1],
                     [1, -1j, -1, 1j]])

def stokes_op(phi):
    return a*b.dag()*np.conj(np.exp(1j*phi)) + b*a.dag()*np.exp(1j*phi)

def make_coeff_from_arr(arr):
    def f(t, args):
        return np.interp(t, args['tlist'], arr)
    return f

steps_def = [

    {'phi': 0, 'tau_factor': 1, 'amps': 2.0,
     'ampp_list': [np.sqrt(2), -np.sqrt(2)],
     'pump_ops': [b*d.dag()+d*b.dag(), b*f.dag()+f*b.dag()]},

    {'phi': np.pi/2, 'tau_factor': -1, 'amps': 2.0,
     'ampp_list': [np.sqrt(2), -np.sqrt(2)],
     'pump_ops': [b*d.dag()+d*b.dag(), b*f.dag()+f*b.dag()]},

    {'phi': 0, 'tau_factor': 1, 'amps': 2.0,
     'ampp_list': [-1.0, 1.0, 1.0, 1.0],
     'pump_ops': [b*c.dag()+c*b.dag(), b*d.dag()+d*b.dag(),
                  b*e.dag()+e*b.dag(), b*f.dag()+f*b.dag()]},

    {'phi': np.pi, 'tau_factor': -1, 'amps': 2.0,
     'ampp_list': [-1.0, 1.0, 1.0, 1.0],
     'pump_ops': [b*c.dag()+c*b.dag(), b*d.dag()+d*b.dag(),
                  b*e.dag()+e*b.dag(), b*f.dag()+f*b.dag()]}
]

def evolve_one_step(psi0, step_idx, T):
    step = steps_def[step_idx]
    phi = step['phi']
    tau = step['tau_factor'] * (T / 0.8)
    amps = step['amps']
    ampp_list = step['ampp_list']
    pump_ops = step['pump_ops']

    psi_shift = -(2*step_idx + 1) * 5 * T
    t_start = step_idx * 10 * T
    t_end = (step_idx + 1) * 10 * T
    tstep = np.linspace(t_start, t_end, N_per_step)

    S_half = 0.5 * amps * np.exp(-(tstep + tau + psi_shift)**2 / T**2)
    P_half_list = [0.5 * amp * np.exp(-(tstep + psi_shift)**2 / T**2) for amp in ampp_list]

    H_terms = []
    H_terms.append([stokes_op(phi), make_coeff_from_arr(S_half)])
    for k in range(len(pump_ops)):
        H_terms.append([pump_ops[k], make_coeff_from_arr(P_half_list[k])])

    args = {'tlist': tstep}
    result = mesolve(H_terms, psi0, tstep, [], args=args, options=solver_opts(T))
    return result.states[-1]

def run_full_sequence(psi0, T):
    psi = psi0
    for step_idx in range(4):
        psi = evolve_one_step(psi, step_idx, T)
    return psi

def compute_gate_fidelity(T):
    comp_basis = [c, d, e, f]
    final_states = [run_full_sequence(ket, T) for ket in comp_basis]

    U_impl = np.zeros((4,4), dtype=complex)
    for j, fket in enumerate(final_states):
        for i, bra_ket in enumerate(comp_basis):
            U_impl[i,j] = bra_ket.dag() * fket

    M = F4.conj().T @ U_impl
    F_avg = (np.trace(M @ M.conj().T).real + np.abs(np.trace(M))**2) / (quartit_dim * (quartit_dim + 1))

    leakage_total = 0.0
    for j in range(4):
        pop_in_comp = np.sum(np.abs(U_impl[:, j])**2)
        leakage_total += (1.0 - pop_in_comp)
    avg_leakage = leakage_total / 4.0
    unitary_err = np.max(np.abs(U_impl.conj().T @ U_impl - np.eye(4)))
    return F_avg, avg_leakage, unitary_err, U_impl

T_values = np.arange(1, 301, 0.5)
n_T = len(T_values)

fidelities = np.zeros(n_T)
avg_leakages = np.zeros(n_T)
unitary_errors = np.zeros(n_T)

print(f"扫描 {n_T} 个 T 值 (quartit STIRAP, only tau = T/0.8)...")
start_time = time.time()

for idx, T in enumerate(tqdm(T_values, desc="扫描 quartit T")):
    F_avg, leak, uerr, _ = compute_gate_fidelity(T)
    fidelities[idx] = F_avg
    avg_leakages[idx] = leak
    unitary_errors[idx] = uerr

total_time = time.time() - start_time
print(f"扫描完成，总用时 {total_time:.2f} 秒")

first_099 = first_0999 = None
for i, fval in enumerate(fidelities):
    if fval >= 0.99 and first_099 is None:
        first_099 = T_values[i]
    if fval >= 0.999 and first_0999 is None:
        first_0999 = T_values[i]

print(f"首次达到 0.99: T = {first_099}")
print(f"首次达到 0.999: T = {first_0999}")
print(f"最高保真度: {fidelities.max():.8f} at T = {T_values[fidelities.argmax()]}")

np.savetxt('quartit_STIRAP_T08_noCD.csv',
           np.column_stack((T_values, fidelities, avg_leakages, unitary_errors)),
           delimiter=',', header='T,AGF,Avg_Leakage,Unitary_Error', comments='')

plt.figure(figsize=(10,6))
plt.plot(T_values, fidelities, 'b-', linewidth=2)
plt.axhline(0.99, color='r', linestyle='--', alpha=0.7)
plt.axhline(0.999, color='g', linestyle='--', alpha=0.7)
plt.xlabel('T')
plt.ylabel('Average Gate Fidelity')
plt.title('Quartit QFT via STIRAP (no CD, tau = T/0.8)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('quartit_STIRAP_T08_noCD_AGF.png', dpi=200)
plt.show()


## 7. Pareto

In [ ]:
def compute_metrics(psi0_list, T, steps, waveform_func, dim):
    final_states = []
    omega_peak = 0.0
    total_energy = 0.0
    total_area = 0.0

    for i_init, init_ket in enumerate(psi0_list):
        psi_final, all_Omega_S, all_Omega_P, all_Omega_CD, t_list = run_protocol(
            init_ket, T, steps, waveform_func)

        if i_init == 0:
            for i_step in range(len(all_Omega_S)):
                Omega_S_step = all_Omega_S[i_step]
                Omega_P_step = all_Omega_P[i_step]
                Omega_CD_step = all_Omega_CD[i_step]
                t_step = t_list[i_step]

                current_peak = np.max(np.abs(Omega_S_step))
                integrand = Omega_S_step**2

                for op_arr in Omega_P_step:
                    current_peak = max(current_peak, np.max(np.abs(op_arr)))
                    integrand += op_arr**2

                for op_arr in Omega_CD_step:
                    current_peak = max(current_peak, np.max(np.abs(op_arr)))
                    integrand += op_arr**2

                omega_peak = max(omega_peak, current_peak)
                total_energy += np.trapz(integrand, t_step)
                total_area += np.trapz(np.sqrt(integrand), t_step)

        final_states.append(psi_final)

    U_impl = np.zeros((dim, dim), dtype=complex)
    comp_basis = psi0_list
    for j, fket in enumerate(final_states):
        for i, bra_ket in enumerate(comp_basis):
            U_impl[i, j] = bra_ket.dag() * fket

    M = F4.conj().T @ U_impl
    F_avg = (np.trace(M @ M.conj().T).real + np.abs(np.trace(M))**2) \
            / (dim * (dim + 1))

    return F_avg, omega_peak, total_energy, total_area

protocols = {
    'qubit_STIRAP': {'steps': qubit_STIRAP_steps, 'waveform': get_STIRAP_waveforms},
    'quartit_STIRAP': {'steps': quartit_STIRAP_steps, 'waveform': get_STIRAP_waveforms},

    'qubit_alloptical': {'steps': qubit_alloptical_steps, 'waveform': get_all_optical_waveforms},
    'quartit_alloptical': {'steps': quartit_alloptical_steps, 'waveform': get_all_optical_waveforms},

    'qubit_STIRSAP': {'steps': qubit_STIRSAP_steps, 'waveform': get_STIRSAP_waveforms},
    'quartit_STIRSAP': {'steps': quartit_STIRSAP_steps, 'waveform': get_STIRSAP_waveforms}
}
basis_states = [basis(6, 2), basis(6, 3), basis(6, 4), basis(6, 5)]

results = []

for prot_name, prot in protocols.items():
    print(f"\n========== 扫描 {prot_name} ==========")
    steps = prot['steps']
    waveform = prot['waveform']

    for T in tqdm(T_values, desc=prot_name):
        F_avg, peak, energy, area = compute_metrics(basis_states, T, steps, waveform, dim=4)
        results.append([prot_name, T, F_avg, peak, energy, area])

results = np.array(results, dtype=object)
np.savetxt('all_protocols_scan.csv', results, delimiter=',', fmt='%s',
           header='Protocol,T,AGF,Omega_peak,Energy,PulseArea', comments='')
print("\n完整扫描数据已保存至 all_protocols_scan.csv")

data_matrix = np.column_stack((
    -results[:, 2].astype(float),
    results[:, 3].astype(float),
    results[:, 4].astype(float)
))

n_points = len(data_matrix)
pareto_mask = np.ones(n_points, dtype=bool)
for i in range(n_points):
    for j in range(n_points):
        if i == j:
            continue
        if (np.all(data_matrix[j] <= data_matrix[i]) and
            np.any(data_matrix[j] < data_matrix[i])):
            pareto_mask[i] = False
            break

pareto_results = results[pareto_mask]
np.savetxt('pareto_non_dominated.csv', pareto_results, delimiter=',', fmt='%s',
           header='Protocol,T,AGF,Omega_peak,Energy,PulseArea', comments='')
print(f"非支配解数量: {len(pareto_results)}，已保存至 pareto_non_dominated.csv")

print("\n非支配解明细：")
for row in pareto_results:
    print(f"{row[0]:20s}  T={row[1]:5.1f}  AGF={float(row[2]):.6f}  Peak={float(row[3]):.3f}  Energy={float(row[4]):.2f}")


In [ ]:
import pandas as pd, numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
%matplotlib inline
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial'],          # Windows 自带
    'axes.unicode_minus': False,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'mathtext.fontset': 'custom',
    'mathtext.rm': 'Arial',
    'mathtext.it': 'Arial:italic',
    'mathtext.default': 'it'
})

df=pd.read_csv('all_protocols_scan.csv'); df.columns=[c.strip() for c in df.columns]
prot=df.iloc[:,0].astype(str).values
T =pd.to_numeric(df.iloc[:,1],errors='coerce').values
AGF=pd.to_numeric(df.iloc[:,2],errors='coerce').values
PK=pd.to_numeric(df.iloc[:,3],errors='coerce').values
EN=pd.to_numeric(df.iloc[:,4],errors='coerce').values
AGI=1-AGF
enc=np.array(['quartit' if p.startswith('quartit') else 'qubit' for p in prot])
sub=np.array(['_'.join(p.split('_')[1:]) for p in prot])

M=np.column_stack([AGI,PK,EN]); n=len(M); par=np.ones(n,bool)
for i in range(n):
    for j in range(n):
        if i!=j and np.all(M[j]<=M[i]) and np.any(M[j]<M[i]): par[i]=False; break

CM=1/2.54; FS=16; AXLW=1.0; DLW=1.5; TKLEN=2.5
CMAP=plt.cm.plasma; VMIN,VMAX=2.0,3.6
MK={'STIRAP':'s','STIRSAP':'o','alloptical':'^'}
ALPHA={'qubit':0.35,'quartit':1.0}

fig=plt.figure(figsize=(8.6*CM,6.7*CM))
ax=fig.add_axes([0.235,0.205,0.545,0.665])

def plot_pts(a_, mask, filled, size, edge_only=False):
    for s,mk in MK.items():
        for e,al in ALPHA.items():
            sel=mask&(sub==s)&(enc==e)
            if not sel.any(): continue
            if filled:
                a_.scatter(EN[sel],AGI[sel],s=size,marker=mk,
                           c=np.clip(PK[sel],VMIN,VMAX),cmap=CMAP,vmin=VMIN,vmax=VMAX,
                           edgecolors='none',alpha=al,zorder=6)
            else:

                a_.scatter(EN[sel],AGI[sel],s=size,marker='D',
                           facecolors='none',edgecolors='#999999',
                           linewidths=0.7,alpha=al,zorder=2)

plot_pts(ax, ~par, filled=False, size=16)

plot_pts(ax, par, filled=True, size=52)

ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlim(30,5e3); ax.set_ylim(4e-9,2.2)
ax.set_xlabel(r'$\Omega^{2}\mathrm{T}$',fontsize=FS,labelpad=1)
ax.set_ylabel('AGI',fontsize=FS,labelpad=-2)
ax.tick_params(which='major',direction='in',length=TKLEN,width=AXLW,top=True,right=True,labelsize=FS)
ax.tick_params(which='minor',direction='in',length=TKLEN*0.6,width=AXLW,top=True,right=True)
ax.set_xticks([1e2,1e3,1e4]); ax.set_xticklabels([r'$10^{2}$',r'$10^{3}$',r'$10^{4}$'])
ax.set_yticks([1e-8,1e-4,1e0]); ax.set_yticklabels([r'$10^{-8}$',r'$10^{-4}$',r'$10^{0}$'])

from matplotlib.ticker import LogLocator
ax.xaxis.set_minor_locator(LogLocator(base=10,subs=np.arange(2,10)*0.1,numticks=100))
ax.yaxis.set_minor_locator(LogLocator(base=10,subs=np.arange(2,10)*0.1,numticks=100))
for sp in ax.spines.values(): sp.set_linewidth(AXLW)

axin=ax.inset_axes([0.02,0.02,0.30,0.30])

for t in [7,10,11,12]:
    ms_=par&(sub=='STIRSAP')&(np.isclose(T,t))
    ma_=par&(sub=='alloptical')&(np.isclose(T,t))
    if ms_.any() and ma_.any():
        axin.plot([EN[ms_][0],EN[ma_][0]],[AGI[ms_][0],AGI[ma_][0]],'-',
                  color='k',lw=0.8,zorder=2)
IN_X0, IN_X1 = 230, 580
for s in ['STIRSAP','alloptical']:
    sel = par & (sub==s) & (EN>IN_X0) & (EN<IN_X1)     
    axin.scatter(EN[sel], AGI[sel], s=30, marker=MK[s],
                 c=np.clip(PK[sel],VMIN,VMAX), cmap=CMAP, vmin=VMIN, vmax=VMAX,
                 edgecolors='none', zorder=6)
axin.set_xlim(IN_X0, IN_X1)
axin.set_ylim(7.8e-6, 1.7e-5)
axin.tick_params(which='both',direction='in',length=TKLEN*0.7,width=AXLW,
                 top=True,right=True,labelleft=False,labelbottom=False,colors='k')
for sp in axin.spines.values(): sp.set_linewidth(AXLW); sp.set_edgecolor('k')
ax.indicate_inset_zoom(axin,edgecolor='k',linewidth=0.8,alpha=0.7)

from matplotlib.ticker import AutoMinorLocator

CB_BOT=0.205; CB_TOP=0.87
cax=fig.add_axes([0.815,CB_BOT,0.045,CB_TOP-CB_BOT])
sm=plt.cm.ScalarMappable(cmap=CMAP,norm=plt.Normalize(VMIN,VMAX))
cb=fig.colorbar(sm,cax=cax)
cb.set_ticks([2.0,2.8,3.6])
cb.minorticks_on()
cb.ax.yaxis.set_minor_locator(AutoMinorLocator(4))
cb.ax.tick_params(which='major',direction='out',length=TKLEN,width=AXLW,labelsize=FS,
                  left=False,right=True)
cb.ax.tick_params(which='minor',direction='out',length=TKLEN*0.6,width=AXLW,
                  left=False,right=True)
cb.outline.set_linewidth(AXLW)

fig.text(0.8375, 0.955, r'$\Omega_\mathrm{max}$', fontsize=FS,
         ha='center', va='center')

fig.savefig('pareto.pdf',dpi=300)
fig.savefig('pareto_preview.png',dpi=300)
plt.show()
print('pareto done')


## 8. amplitude+pure dephasing

In [ ]:
import numpy as np
from qutip import *
import matplotlib.pyplot as plt

np.seterr(divide='ignore', invalid='ignore')

T_NZ          = 7.5
N_NZ = 1000
N_SAMP_NZ  = 50
SEED       = None
sigma_sg, p_sg = 3.0, 10

n_eps, n_gamma = 50, 50
epsilon_vals = np.linspace(-0.10, 0.10, n_eps)
gammaT_vals  = np.logspace(-6, -1, n_gamma)

import qutip as _qt
_QT5 = int(_qt.__version__.split('.')[0]) >= 5
def make_opts(od):
    return dict(od) if _QT5 else Options(**od)
OPTS      = make_opts(dict(nsteps=100000, atol=1e-10, rtol=1e-8,  max_step=T_NZ/10))
OPTS_FINE = make_opts(dict(nsteps=200000, atol=1e-12, rtol=1e-10, max_step=T_NZ/20))

a = basis(6, 0); b = basis(6, 1)
c = basis(6, 2); d = basis(6, 3); e = basis(6, 4); f = basis(6, 5)

QFT4 = 0.5 * np.array([[1, 1, 1, 1],
                       [1, 1j, -1, -1j],
                       [1, -1, 1, -1],
                       [1, -1j, -1, 1j]])

J_z = Qobj(np.diag([0, 0, -1.5, -0.5, 0.5, 1.5]))

def spontaneous_emission_ops(gamma):
    if gamma == 0:
        return []
    r = np.sqrt(gamma / 5.0)
    return [r*a*b.dag(), r*c*b.dag(), r*d*b.dag(), r*e*b.dag(), r*f*b.dag()]

def stokes_op(phi):
    return a*b.dag()*np.conj(np.exp(1j*phi)) + b*a.dag()*np.exp(1j*phi)

STEPS_NZ = [
    {'phi': 0,       'tf':  1, 'ampp': [np.sqrt(2), -np.sqrt(2)],
     'ops': [b*d.dag()+d*b.dag(), b*f.dag()+f*b.dag()]},
    {'phi': np.pi/2, 'tf': -1, 'ampp': [np.sqrt(2), -np.sqrt(2)],
     'ops': [b*d.dag()+d*b.dag(), b*f.dag()+f*b.dag()]},
    {'phi': 0,       'tf':  1, 'ampp': [-1.0, 1.0, 1.0, 1.0],
     'ops': [b*c.dag()+c*b.dag(), b*d.dag()+d*b.dag(),
             b*e.dag()+e*b.dag(), b*f.dag()+f*b.dag()]},
    {'phi': np.pi,   'tf': -1, 'ampp': [-1.0, 1.0, 1.0, 1.0],
     'ops': [b*c.dag()+c*b.dag(), b*d.dag()+d*b.dag(),
             b*e.dag()+e*b.dag(), b*f.dag()+f*b.dag()]},
]

def stirap_waveforms(tl, tau, ampp, shift, T_val):
    S = 0.5 * 2.0 * np.exp(-(tl + tau + shift)**2 / T_val**2)
    P = [0.5 * amp * np.exp(-(tl + shift)**2 / T_val**2) for amp in ampp]
    w = np.exp(-(np.abs(tl + shift) / (sigma_sg * T_val))**p_sg)
    return S * w, [p * w for p in P]

def alloptical_waveforms(tl, tau, ampp, shift, T_val):
    dt = tl[1] - tl[0]
    S_raw = 0.5 * 2.0 * np.exp(-(tl + tau + shift)**2 / T_val**2)
    P_raw = [0.5 * amp * np.exp(-(tl + shift)**2 / T_val**2) for amp in ampp]

    OS  = 2.0 * S_raw
    OPl = [2.0 * p for p in P_raw]
    OP  = np.sqrt(sum(p**2 for p in OPl))

    dOP = np.gradient(OP, dt); dOS = np.gradient(OS, dt)
    den = OP**2 + OS**2
    thd = np.divide(dOP*OS - dOS*OP, den, where=(den >= 1e-12),
                    out=np.zeros_like(OS))

    w  = np.exp(-(np.abs(tl + shift) / (sigma_sg * T_val))**p_sg)
    Oa = 2.0 * thd * w

    phr  = np.arctan2(Oa, OP)
    dphr = np.gradient(phr, dt)

    OPt = np.sqrt(OP**2 + Oa**2)
    OSt = OS - 2.0 * dphr

    sc = np.ones_like(OP); m = OP > 1e-12
    sc[m] = OPt[m] / OP[m]
    return OSt / 2.0, [0.5 * p * sc for p in OPl]

WAVEFORMS = {'STIRAP': stirap_waveforms, 'alloptical': alloptical_waveforms}

def run_seq_nz(rho0, wf_name, epsilon, c_ops, opts=None, N=None):
    opts = OPTS if opts is None else opts
    N = N_NZ if N is None else N
    wf = WAVEFORMS[wf_name]
    rho = rho0
    for k, st in enumerate(STEPS_NZ):
        tau   = st['tf'] * (T_NZ / 0.8)
        shift = -(2*k + 1) * 5 * T_NZ
        tl    = np.linspace(k*10*T_NZ, (k+1)*10*T_NZ, N)

        S, P = wf(tl, tau, st['ampp'], shift, T_NZ)
        S = (1 + epsilon) * S
        P = [(1 + epsilon) * p for p in P]

        H = [[stokes_op(st['phi']),
              (lambda t, args=None, arr=S, tt=tl: np.interp(t, tt, arr))]]
        for kk, op in enumerate(st['ops']):
            H.append([op, (lambda t, args=None, arr=P[kk], tt=tl: np.interp(t, tt, arr))])

        rho = mesolve(H, rho, tl, c_ops, options=opts).states[-1]

        tr = np.real(rho.tr())
        assert abs(tr - 1.0) < 1e-6, \
            f"solver failure: Tr={tr:.6f} at step {k}, eps={epsilon}"
    return rho

rng = np.random.default_rng(SEED)
SAMPLES = []
for _ in range(N_SAMP_NZ):
    v = rng.standard_normal(4) + 1j * rng.standard_normal(4)
    v /= np.linalg.norm(v)
    p0 = np.zeros(6, complex); p0[2:] = v
    tv = QFT4 @ v; tv /= np.linalg.norm(tv)
    pt = np.zeros(6, complex); pt[2:] = tv
    SAMPLES.append((ket2dm(Qobj(p0)), Qobj(pt)))

def agf_dephasing(wf_name, epsilon, gamma_T, opts=None, N=None, samples=None):
    gamma = gamma_T / (40 * T_NZ) if gamma_T > 0 else 0.0
    c_ops = [np.sqrt(gamma) * J_z] if gamma > 0 else []
    ss = SAMPLES if samples is None else samples
    return float(np.mean([np.real(expect(
        run_seq_nz(rho0, wf_name, epsilon, c_ops, opts, N), psi_t))
        for rho0, psi_t in ss]))

def agf_emission(wf_name, gamma_T, opts=None, N=None):
    gamma = gamma_T / (40 * T_NZ) if gamma_T > 0 else 0.0
    c_ops = spontaneous_emission_ops(gamma)
    return float(np.mean([np.real(expect(
        run_seq_nz(rho0, wf_name, 0.0, c_ops, opts, N), psi_t))
        for rho0, psi_t in SAMPLES]))

def audit_convergence(n_check=6, seed=None):
    r = np.random.default_rng(seed)
    sub = SAMPLES[:10]
    worst = 0.0
    for _ in range(n_check):
        wf  = r.choice(['STIRAP', 'alloptical'])
        eps = r.choice(epsilon_vals)
        gT  = r.choice(gammaT_vals)
        F1 = agf_dephasing(wf, eps, gT, OPTS,      N_NZ,   sub)
        F2 = agf_dephasing(wf, eps, gT, OPTS_FINE, 2*N_NZ, sub)
        print(f"  {wf:10s} eps={eps:+.3f} gT={gT:.1e}: "
              f"F={F1:.7f} vs fine {F2:.7f}  |dF|={abs(F1-F2):.1e}")
        worst = max(worst, abs(F1 - F2))
    print(f"audit: max |dF| = {worst:.2e}")
    return worst

print("公共模块已加载：T_NZ =", T_NZ, "| N_SAMP_NZ =", N_SAMP_NZ, "| qutip", _qt.__version__)


In [ ]:
import numpy as np
from qutip import *
import matplotlib.pyplot as plt
from tqdm import tqdm
from joblib import Parallel, delayed
import time

# ==================== 固定参数 ====================
T = 7.5                    
N_per_step = 1000
n_samples = 100
N_jobs = -1

# 噪声网格
n_eps = 50
n_gamma = 50
epsilon_vals = np.linspace(-0.10, 0.10, n_eps)
gammaT_vals = np.logspace(-6, -1, n_gamma)

# ==================== 超高斯窗参数 ====================

import qutip as _qt
_QT5 = int(_qt.__version__.split('.')[0]) >= 5
_od = dict(nsteps=100000, atol=1e-10, rtol=1e-8, max_step=T/10)
SOLVER_OPTS = _od if _QT5 else Options(**_od)
sigma_sg = 3.0
p_sg = 10

# ==================== 基矢与 QFT 门 ====================
a = basis(6, 0)
b = basis(6, 1)
c = basis(6, 2)   # |0>
d = basis(6, 3)   # |1>
e = basis(6, 4)   # |2>
f = basis(6, 5)   # |3>

QFT4 = 0.5 * np.array([[1, 1, 1, 1],
                       [1, 1j, -1, -1j],
                       [1, -1, 1, -1],
                       [1, -1j, -1, 1j]])

J_z = Qobj(np.diag([0, 0, -1.5, -0.5, 0.5, 1.5]))

# ==================== 原始高斯脉冲生成（加超高斯窗截断 + 振幅噪声） ====================
def get_raw_gaussian_pulses_sg(tlist, tau, amps, ampp_list, psi, T_val, epsilon):
    """
    生成原始 Stokes 和 Pump 半幅值，并乘以超高斯窗实现平滑截断。
    脉冲间隔使用 T/0.8（由 tau 体现，此处不做额外修改）。
    """
    amps_scaled = amps * (1 + epsilon)
    ampp_list_scaled = [amp * (1 + epsilon) for amp in ampp_list]

    S_half = 0.5 * amps_scaled * np.exp(-(tlist + tau + psi)**2 / T_val**2)
    P_half_list = [0.5 * amp * np.exp(-(tlist + psi)**2 / T_val**2) for amp in ampp_list_scaled]

    # 施加超高斯窗
    tc = -psi
    window = np.exp(-(np.abs(tlist - tc) / (sigma_sg * T_val))**p_sg)
    S_half *= window
    P_half_list = [p * window for p in P_half_list]

    return S_half, P_half_list

# ==================== 算符 ====================
def stokes_op(phi):
    return a * b.dag() * np.conj(np.exp(1j*phi)) + b * a.dag() * np.exp(1j*phi)

def make_coeff_from_arr(arr):
    def f(t, args):
        return np.interp(t, args['tlist'], arr)
    return f

# ==================== 四步 QHR 参数（无 CD） ====================
steps_def = [
    {'phi': 0, 'tau_factor': 1, 'amps': 2.0,
     'ampp_list': [np.sqrt(2), -np.sqrt(2)],
     'pump_ops': [b*d.dag()+d*b.dag(), b*f.dag()+f*b.dag()]},
    {'phi': np.pi/2, 'tau_factor': -1, 'amps': 2.0,
     'ampp_list': [np.sqrt(2), -np.sqrt(2)],
     'pump_ops': [b*d.dag()+d*b.dag(), b*f.dag()+f*b.dag()]},
    {'phi': 0, 'tau_factor': 1, 'amps': 2.0,
     'ampp_list': [-1.0, 1.0, 1.0, 1.0],
     'pump_ops': [b*c.dag()+c*b.dag(), b*d.dag()+d*b.dag(),
                  b*e.dag()+e*b.dag(), b*f.dag()+f*b.dag()]},
    {'phi': np.pi, 'tau_factor': -1, 'amps': 2.0,
     'ampp_list': [-1.0, 1.0, 1.0, 1.0],
     'pump_ops': [b*c.dag()+c*b.dag(), b*d.dag()+d*b.dag(),
                  b*e.dag()+e*b.dag(), b*f.dag()+f*b.dag()]}
]

# ==================== 单步演化（密度矩阵，含退相干） ====================
def evolve_one_step(rho0, step_idx, T_val, epsilon, gamma):
    step = steps_def[step_idx]
    phi_s = step['phi']
    # 脉冲间隔改为 T/0.8
    tau = step['tau_factor'] * (T_val / 0.8)
    amps = step['amps']
    ampp_list = step['ampp_list']
    pump_ops = step['pump_ops']

    psi_shift = -(2*step_idx + 1) * 5 * T_val
    t_start = step_idx * 10 * T_val
    t_end = (step_idx + 1) * 10 * T_val
    tstep = np.linspace(t_start, t_end, N_per_step)

    S, P_list = get_raw_gaussian_pulses_sg(tstep, tau, amps, ampp_list, psi_shift, T_val, epsilon)

    H_terms = [[stokes_op(phi_s), make_coeff_from_arr(S)]]
    for k, op in enumerate(pump_ops):
        H_terms.append([op, make_coeff_from_arr(P_list[k])])

    c_ops = [np.sqrt(gamma) * J_z] if gamma > 0 else []
    args = {'tlist': tstep}
    result = mesolve(H_terms, rho0, tstep, c_ops, args=args, options=SOLVER_OPTS)
    return result.states[-1]

def run_full_sequence(rho0, T_val, epsilon, gamma):
    rho = rho0
    for step_idx in range(4):
        rho = evolve_one_step(rho, step_idx, T_val, epsilon, gamma)
    return rho

# ==================== 平均保真度 ====================
def average_fidelity_one_config(epsilon, gamma_T):
    total_time = 40 * T
    gamma = gamma_T / total_time if gamma_T > 0 else 0.0
    fid_sum = 0.0

    for _ in range(n_samples):
        rand_vec = np.random.randn(4) + 1j * np.random.randn(4)
        rand_vec /= np.linalg.norm(rand_vec)
        psi_init = np.zeros(6, dtype=complex)
        psi_init[2:6] = rand_vec
        rho0 = Qobj(psi_init) * Qobj(psi_init).dag()

        psi_tgt_comp = QFT4 @ rand_vec
        psi_tgt_comp /= np.linalg.norm(psi_tgt_comp)
        psi_tgt_6d = np.zeros(6, dtype=complex)
        psi_tgt_6d[2:6] = psi_tgt_comp
        rho_target = Qobj(psi_tgt_6d) * Qobj(psi_tgt_6d).dag()

        rho_final = run_full_sequence(rho0, T, epsilon, gamma)

        if gamma == 0:
            eigvals, eigvecs = np.linalg.eig(rho_final.full())
            max_idx = np.argmax(np.real(eigvals))
            psi_final = Qobj(eigvecs[:, max_idx])
            state_fid = np.abs((psi_tgt_6d.conj().T @ psi_final.full()).flatten()[0])**2
        else:
            state_fid = fidelity(rho_target, rho_final)

        fid_sum += state_fid
    return fid_sum / n_samples

# ==================== 主程序 ====================
if __name__ == "__main__":
    param_list = [(eps, gT) for eps in epsilon_vals for gT in gammaT_vals]
    print(f"Total configurations: {len(param_list)}")
    start = time.time()
    results = Parallel(n_jobs=N_jobs, prefer="processes", verbose=0)(
        delayed(average_fidelity_one_config)(eps, gT) for eps, gT in tqdm(param_list, desc="STIRAP scan")
    )
    print(f"Scan completed in {time.time()-start:.1f} s")

    AGF_matrix = np.array(results).reshape(n_eps, n_gamma)

    csv_data = np.column_stack((
        np.repeat(epsilon_vals, n_gamma),
        np.tile(gammaT_vals, n_eps),
        AGF_matrix.ravel()
    ))
    np.savetxt("STIRAP_noCD_T7.5_SGwindow_noise.csv", csv_data,
               delimiter=',', header='epsilon,gamma_T,AGF', comments='')
    print("Results saved to STIRAP_noCD_T7.5_SGwindow_noise.csv")

    plt.figure(figsize=(8, 6))
    X, Y = np.meshgrid(gammaT_vals, epsilon_vals)
    plt.pcolormesh(X, Y, AGF_matrix, shading='auto', cmap='viridis')
    plt.colorbar(label='AGF')
    plt.xscale('log')
    plt.xlabel(r'$\gamma T$')
    plt.ylabel(r'$\epsilon$')
    plt.title('STIRAP (no CD), T=7.5, SG window on pulses + τ = T/0.8')
    plt.tight_layout()
    plt.savefig('STIRAP_noCD_T7.5_SGwindow_noise.png', dpi=200)
    plt.show()

In [ ]:
import numpy as np
from qutip import *
import matplotlib.pyplot as plt
from tqdm import tqdm
from joblib import Parallel, delayed
import time

T = 7.5 
N_per_step = 1000
n_samples = 100
N_jobs = -1

n_eps = 50
n_gamma = 50
epsilon_vals = np.linspace(-0.10, 0.10, n_eps)
gammaT_vals = np.logspace(-6, -1, n_gamma)

sigma_sg = 3.0
p_sg = 10

a = basis(6, 0)
b = basis(6, 1)
c = basis(6, 2) 
d = basis(6, 3) 
e = basis(6, 4) 
f = basis(6, 5) 

QFT4 = 0.5 * np.array([[1, 1, 1, 1],
                       [1, 1j, -1, -1j],
                       [1, -1, 1, -1],
                       [1, -1j, -1, 1j]])

J_z = Qobj(np.diag([0, 0, -1.5, -0.5, 0.5, 1.5]))

def get_all_optical_waveforms(tlist, tau, amps, ampp_list, psi, T_val, epsilon):
    dt = tlist[1] - tlist[0]
    eps = 1e-12

    amps_scaled = amps * (1 + epsilon)
    ampp_list_scaled = [amp * (1 + epsilon) for amp in ampp_list]

    S_half_raw = 0.5 * amps_scaled * np.exp(-(tlist + tau + psi)**2 / T_val**2)
    P_half_raw_list = [0.5 * amp * np.exp(-(tlist + psi)**2 / T_val**2) for amp in ampp_list_scaled]

    Omega_S = 2.0 * S_half_raw
    Omega_P_list = [2.0 * p for p in P_half_raw_list]
    Omega_P = np.sqrt(sum(p**2 for p in Omega_P_list))

    dOmega_P = np.gradient(Omega_P, dt)
    dOmega_S = np.gradient(Omega_S, dt)

    denom = Omega_P**2 + Omega_S**2
    theta_dot = np.divide(dOmega_P * Omega_S - dOmega_S * Omega_P,
                          denom,
                          where=(denom >= eps),
                          out=np.zeros_like(Omega_S))

    tc = -psi
    window = np.exp(-(np.abs(tlist - tc) / (sigma_sg * T_val))**p_sg)
    Omega_a = 2.0 * theta_dot * window

    phi_rot = np.arctan2(Omega_a, Omega_P)
    dphi_rot = np.gradient(phi_rot, dt)

    Omega_P_tilde = np.sqrt(Omega_P**2 + Omega_a**2)
    Omega_S_tilde = Omega_S - 2.0 * dphi_rot

    scale = np.ones_like(Omega_P)
    mask_P = Omega_P > eps
    scale[mask_P] = Omega_P_tilde[mask_P] / Omega_P[mask_P]

    S_half_mod = Omega_S_tilde / 2.0
    P_half_mod_list = [0.5 * p * scale for p in Omega_P_list]

    return S_half_mod, P_half_mod_list

def stokes_op(phi):
    return a * b.dag() * np.conj(np.exp(1j*phi)) + b * a.dag() * np.exp(1j*phi)

def make_coeff_from_arr(arr):
    def f(t, args):
        return np.interp(t, args['tlist'], arr)
    return f

steps_def = [
    {'phi': 0, 'tau_factor': 1, 'amps': 2.0,
     'ampp_list': [np.sqrt(2), -np.sqrt(2)],
     'pump_ops': [b*d.dag()+d*b.dag(), b*f.dag()+f*b.dag()]},
    {'phi': np.pi/2, 'tau_factor': -1, 'amps': 2.0,
     'ampp_list': [np.sqrt(2), -np.sqrt(2)],
     'pump_ops': [b*d.dag()+d*b.dag(), b*f.dag()+f*b.dag()]},
    {'phi': 0, 'tau_factor': 1, 'amps': 2.0,
     'ampp_list': [-1.0, 1.0, 1.0, 1.0],
     'pump_ops': [b*c.dag()+c*b.dag(), b*d.dag()+d*b.dag(),
                  b*e.dag()+e*b.dag(), b*f.dag()+f*b.dag()]},
    {'phi': np.pi, 'tau_factor': -1, 'amps': 2.0,
     'ampp_list': [-1.0, 1.0, 1.0, 1.0],
     'pump_ops': [b*c.dag()+c*b.dag(), b*d.dag()+d*b.dag(),
                  b*e.dag()+e*b.dag(), b*f.dag()+f*b.dag()]}
]

def evolve_one_step(rho0, step_idx, T_val, epsilon, gamma):
    step = steps_def[step_idx]
    phi_s = step['phi']
    tau = step['tau_factor'] * (T_val / 0.8)
    amps = step['amps']
    ampp_list = step['ampp_list']
    pump_ops = step['pump_ops']

    psi_shift = -(2*step_idx + 1) * 5 * T_val
    t_start = step_idx * 10 * T_val
    t_end = (step_idx + 1) * 10 * T_val
    tstep = np.linspace(t_start, t_end, N_per_step)

    S_mod, P_mod_list = get_all_optical_waveforms(tstep, tau, amps, ampp_list, psi_shift, T_val, epsilon)

    H_terms = [[stokes_op(phi_s), make_coeff_from_arr(S_mod)]]
    for k, op in enumerate(pump_ops):
        H_terms.append([op, make_coeff_from_arr(P_mod_list[k])])

    c_ops = [np.sqrt(gamma) * J_z] if gamma > 0 else []
    args = {'tlist': tstep}
    result = mesolve(H_terms, rho0, tstep, c_ops, args=args)
    return result.states[-1]

def run_full_sequence(rho0, T_val, epsilon, gamma):
    rho = rho0
    for step_idx in range(4):
        rho = evolve_one_step(rho, step_idx, T_val, epsilon, gamma)
    return rho

def average_fidelity_one_config(epsilon, gamma_T):
    total_time = 40 * T
    gamma = gamma_T / total_time if gamma_T > 0 else 0.0
    fid_sum = 0.0

    for _ in range(n_samples):
        rand_vec = np.random.randn(4) + 1j * np.random.randn(4)
        rand_vec /= np.linalg.norm(rand_vec)
        psi_init = np.zeros(6, dtype=complex)
        psi_init[2:6] = rand_vec
        rho0 = Qobj(psi_init) * Qobj(psi_init).dag()

        psi_tgt_comp = QFT4 @ rand_vec
        psi_tgt_comp /= np.linalg.norm(psi_tgt_comp)
        psi_tgt_6d = np.zeros(6, dtype=complex)
        psi_tgt_6d[2:6] = psi_tgt_comp
        rho_target = Qobj(psi_tgt_6d) * Qobj(psi_tgt_6d).dag()

        rho_final = run_full_sequence(rho0, T, epsilon, gamma)

        if gamma == 0:
            eigvals, eigvecs = np.linalg.eig(rho_final.full())
            max_idx = np.argmax(np.real(eigvals))
            psi_final = Qobj(eigvecs[:, max_idx])
            state_fid = np.abs((psi_tgt_6d.conj().T @ psi_final.full()).flatten()[0])**2
        else:
            state_fid = fidelity(rho_target, rho_final)

        fid_sum += state_fid
    return fid_sum / n_samples

if __name__ == "__main__":
    param_list = [(eps, gT) for eps in epsilon_vals for gT in gammaT_vals]
    print(f"Total configurations: {len(param_list)}")
    start = time.time()
    results = Parallel(n_jobs=N_jobs, prefer="processes", verbose=0)(
        delayed(average_fidelity_one_config)(eps, gT) for eps, gT in tqdm(param_list, desc="All-optical scan")
    )
    print(f"Scan completed in {time.time()-start:.1f} s")

    AGF_matrix = np.array(results).reshape(n_eps, n_gamma)

    csv_data = np.column_stack((
        np.repeat(epsilon_vals, n_gamma),
        np.tile(gammaT_vals, n_eps),
        AGF_matrix.ravel()
    ))
    np.savetxt("alloptical_T7.5_SGwindow_noise.csv", csv_data,
               delimiter=',', header='epsilon,gamma_T,AGF', comments='')
    print("Results saved to alloptical_T7.5_SGwindow_noise.csv")

    plt.figure(figsize=(8, 6))
    X, Y = np.meshgrid(gammaT_vals, epsilon_vals)
    plt.pcolormesh(X, Y, AGF_matrix, shading='auto', cmap='viridis')
    plt.colorbar(label='AGF')
    plt.xscale('log')
    plt.xlabel(r'$\gamma T$')
    plt.ylabel(r'$\epsilon$')
    plt.title('All-optical, T=7.5, SG window + τ = T/0.8')
    plt.tight_layout()
    plt.savefig('alloptical_T7.5_SGwindow_noise.png', dpi=200)
    plt.show()

## 9. Emission

In [ ]:
import numpy as np
from qutip import *
import matplotlib.pyplot as plt
import time
from tqdm import tqdm

np.seterr(divide='ignore', invalid='ignore')

n_states = 6
quartit_dim = 4
N_per_step = 1000    
n_samples = 100         
T_val = 7.5   

sigma_sg = 3.0             
p_sg = 10                  


a = basis(6, 0)  
b = basis(6, 1)   
c = basis(6, 2)
d = basis(6, 3)  
e = basis(6, 4)  
f = basis(6, 5)  

QFT4 = 0.5 * np.array([[1, 1, 1, 1],
                       [1, 1j, -1, -1j],
                       [1, -1, 1, -1],
                       [1, -1j, -1, 1j]])

def get_spontaneous_emission_ops(gamma):
    if gamma == 0:
        return []
    n_ground = 5  
    rate_each = gamma / n_ground
    sqrt_rate = np.sqrt(rate_each)
    return [
        sqrt_rate * a * b.dag(),
        sqrt_rate * c * b.dag(),
        sqrt_rate * d * b.dag(),
        sqrt_rate * e * b.dag(),
        sqrt_rate * f * b.dag()
    ]


def get_all_optical_waveforms(tlist, tau, amps, ampp_list, psi, T):
    dt = tlist[1] - tlist[0]


    S_half_raw = 0.5 * amps * np.exp(-(tlist + tau + psi)**2 / T**2)
    P_half_raw_list = [0.5 * amp * np.exp(-(tlist + psi)**2 / T**2) for amp in ampp_list]

    Omega_S = 2.0 * S_half_raw
    Omega_P_list = [2.0 * p for p in P_half_raw_list]
    Omega_P = np.sqrt(sum(p**2 for p in Omega_P_list))

    dOmega_P = np.gradient(Omega_P, dt)
    dOmega_S = np.gradient(Omega_S, dt)

    denom = Omega_P**2 + Omega_S**2
    theta_dot = np.divide(dOmega_P * Omega_S - dOmega_S * Omega_P,
                          denom,
                          where=(denom >= 1e-250),
                          out=np.zeros_like(Omega_S))

    tc = -psi  
    super_gaussian_window = np.exp(-(np.abs(tlist - tc) / (sigma_sg * T))**p_sg)
    Omega_a = 2.0 * theta_dot * super_gaussian_window

    phi_rot = np.arctan2(Omega_a, Omega_P)
    dphi_rot = np.gradient(phi_rot, dt)

    Omega_P_tilde = np.sqrt(Omega_P**2 + Omega_a**2)   
    Omega_S_tilde = Omega_S - 2.0 * dphi_rot

    scale = np.divide(Omega_P_tilde, Omega_P, where=(Omega_P > 1e-100), out=np.ones_like(Omega_P))
    S_half_mod = Omega_S_tilde / 2.0
    P_half_mod_list = [0.5 * p * scale for p in Omega_P_list]
    
    return S_half_mod, P_half_mod_list

def stokes_op(phase):
    return a * b.dag() * np.conj(np.exp(1j*phase)) + b * a.dag() * np.exp(1j*phase)

steps_def = [
    {'stokes_phase': 0, 'tau_factor': 1, 'amps': 2.0,
     'ampp_list': [np.sqrt(2), -np.sqrt(2)],
     'pump_ops': [b*d.dag()+d*b.dag(), b*f.dag()+f*b.dag()]},
    {'stokes_phase': np.pi/2, 'tau_factor': -1, 'amps': 2.0,
     'ampp_list': [np.sqrt(2), -np.sqrt(2)],
     'pump_ops': [b*d.dag()+d*b.dag(), b*f.dag()+f*b.dag()]},
    {'stokes_phase': 0, 'tau_factor': 1, 'amps': 2.0,
     'ampp_list': [-1.0, 1.0, 1.0, 1.0],
     'pump_ops': [b*c.dag()+c*b.dag(), b*d.dag()+d*b.dag(),
                  b*e.dag()+e*b.dag(), b*f.dag()+f*b.dag()]},
    {'stokes_phase': np.pi, 'tau_factor': -1, 'amps': 2.0,
     'ampp_list': [-1.0, 1.0, 1.0, 1.0],
     'pump_ops': [b*c.dag()+c*b.dag(), b*d.dag()+d*b.dag(),
                  b*e.dag()+e*b.dag(), b*f.dag()+f*b.dag()]}
]

def make_coeff_from_arr(arr):
    def f(t, args):
        return np.interp(t, args['tlist'], arr)
    return f

def evolve_one_step_rho(rho0, step_idx, T, c_ops):
    step = steps_def[step_idx]
    stokes_phase = step['stokes_phase']    
    tau = step['tau_factor'] * (T / 0.8)
    amps = step['amps']
    ampp_list = step['ampp_list']
    pump_ops = step['pump_ops']

    psi_shift = -(2*step_idx + 1) * 5 * T
    t_start = step_idx * 10 * T
    t_end = (step_idx + 1) * 10 * T
    tstep = np.linspace(t_start, t_end, N_per_step)

    S_mod, P_mod_list = get_all_optical_waveforms(tstep, tau, amps, ampp_list, psi_shift, T)

    H_terms = []
    H_terms.append([stokes_op(stokes_phase), make_coeff_from_arr(S_mod)])  
    for k, op in enumerate(pump_ops):
        H_terms.append([op, make_coeff_from_arr(P_mod_list[k])])

    args = {'tlist': tstep}

    result = mesolve(H_terms, rho0, tstep, c_ops, [], args=args)
    return result.states[-1]

def run_full_sequence(rho0, T, gamma):
    rho = rho0
    c_ops = get_spontaneous_emission_ops(gamma)
    for step_idx in range(4):
        rho = evolve_one_step_rho(rho, step_idx, T, c_ops)
    return rho

def average_fidelity(gamma_T, T_val):
    total_time = 40 * T_val
    gamma = gamma_T / total_time if gamma_T > 0 else 0.0
    fid_sum = 0.0
    
    for _ in range(n_samples):
        rand_vec = np.random.randn(4) + 1j * np.random.randn(4)
        rand_vec /= np.linalg.norm(rand_vec)
        
        psi_init = np.zeros(6, dtype=complex)
        psi_init[2:6] = rand_vec
        rho0 = Qobj(psi_init) * Qobj(psi_init).dag()

        psi_tgt_comp = QFT4 @ rand_vec
        psi_tgt_comp /= np.linalg.norm(psi_tgt_comp)
        psi_tgt_6d = np.zeros(6, dtype=complex)
        psi_tgt_6d[2:6] = psi_tgt_comp
        rho_target = Qobj(psi_tgt_6d) * Qobj(psi_tgt_6d).dag()

        rho_final = run_full_sequence(rho0, T_val, gamma)

        state_fid = np.real(expect(rho_target, rho_final))
        fid_sum += state_fid
        
    return fid_sum / n_samples

num_points = 20
gamma_T_values =[0]+np.logspace(-6, -1, num_points)
fidelities = np.zeros(num_points)

print(f"正在对 T={T_val} 进行自发辐射扫描，抽取 {n_samples} 个哈尔随机态...")
start_time = time.time()

for idx, gT in enumerate(tqdm(gamma_T_values, desc="Scanning gamma*T")):
    fidelities[idx] = average_fidelity(gT, T_val)

total_time = time.time() - start_time
print(f"扫描完成，总用时 {total_time:.2f} 秒")

data = np.column_stack((gamma_T_values, fidelities))
header = "gammaT,AGF"
np.savetxt('quartit_alloptical_spontaneous_emission.csv', data, delimiter=',', header=header, comments='')
print("数据已保存为 quartit_alloptical_spontaneous_emission.csv")

plt.figure(figsize=(8, 6))
plt.plot(gamma_T_values, fidelities, 'o-', color='C0', linewidth=2, label=f'All-optical STIRSAP (T={T_val})')
plt.xscale('log')
plt.xlabel(r'Dimensionless Spontaneous Emission ($\gamma t$)')
plt.ylabel('Average Gate Fidelity (AGF)')
plt.title(f'Effect of Spontaneous Emission on AGF (Monte Carlo over {n_samples} states)')
plt.grid(True, alpha=0.3, which='both')
plt.legend()
plt.tight_layout()
plt.savefig('quartit_alloptical_emission_curve.png', dpi=200)
plt.show()

In [ ]:
import numpy as np
from qutip import *
import matplotlib.pyplot as plt
import time
from tqdm import tqdm

n_states = 6
quartit_dim = 4
N_per_step = 1000     
n_samples = 100       
T_val = 7.5         

a = basis(6, 0) 
b = basis(6, 1)
c = basis(6, 2)  
d = basis(6, 3)
e = basis(6, 4)  
f = basis(6, 5) 

QFT4 = 0.5 * np.array([[1, 1, 1, 1],
                       [1, 1j, -1, -1j],
                       [1, -1, 1, -1],
                       [1, -1j, -1, 1j]])

def get_spontaneous_emission_ops(gamma):
    if gamma == 0:
        return []
    n_ground = 5  
    rate_each = gamma / n_ground
    sqrt_rate = np.sqrt(rate_each)
    return [
        sqrt_rate * a * b.dag(),
        sqrt_rate * c * b.dag(),
        sqrt_rate * d * b.dag(),
        sqrt_rate * e * b.dag(),
        sqrt_rate * f * b.dag()
    ]

def get_STIRAP_waveforms(tlist, tau, amps, ampp_list, psi, T):
    """返回 Stokes 半幅值和 Pump 半幅值列表（无修饰）"""
    S_half = 0.5 * amps * np.exp(-(tlist + tau + psi)**2 / T**2)
    P_half_list = [0.5 * amp * np.exp(-(tlist + psi)**2 / T**2) for amp in ampp_list]
    return S_half, P_half_list

def stokes_op(phase):
    return a * b.dag() * np.conj(np.exp(1j*phase)) + b * a.dag() * np.exp(1j*phase)

steps_def = [
    {'stokes_phase': 0, 'tau_factor': 1, 'amps': 2.0,
     'ampp_list': [np.sqrt(2), -np.sqrt(2)],
     'pump_ops': [b*d.dag()+d*b.dag(), b*f.dag()+f*b.dag()]},
    {'stokes_phase': np.pi/2, 'tau_factor': -1, 'amps': 2.0,
     'ampp_list': [np.sqrt(2), -np.sqrt(2)],
     'pump_ops': [b*d.dag()+d*b.dag(), b*f.dag()+f*b.dag()]},
    {'stokes_phase': 0, 'tau_factor': 1, 'amps': 2.0,
     'ampp_list': [-1.0, 1.0, 1.0, 1.0],
     'pump_ops': [b*c.dag()+c*b.dag(), b*d.dag()+d*b.dag(),
                  b*e.dag()+e*b.dag(), b*f.dag()+f*b.dag()]},
    {'stokes_phase': np.pi, 'tau_factor': -1, 'amps': 2.0,
     'ampp_list': [-1.0, 1.0, 1.0, 1.0],
     'pump_ops': [b*c.dag()+c*b.dag(), b*d.dag()+d*b.dag(),
                  b*e.dag()+e*b.dag(), b*f.dag()+f*b.dag()]}
]

def make_coeff_from_arr(arr):
    def f(t, args):
        return np.interp(t, args['tlist'], arr)
    return f

def evolve_one_step_rho(rho0, step_idx, T, c_ops):
    step = steps_def[step_idx]
    stokes_phase = step['stokes_phase']
    tau = step['tau_factor'] * (T / 0.8)  
    amps = step['amps']
    ampp_list = step['ampp_list']
    pump_ops = step['pump_ops']

    psi_shift = -(2*step_idx + 1) * 5 * T
    t_start = step_idx * 10 * T
    t_end = (step_idx + 1) * 10 * T
    tstep = np.linspace(t_start, t_end, N_per_step)

    S_half, P_half_list = get_STIRAP_waveforms(tstep, tau, amps, ampp_list, psi_shift, T)

    H_terms = []
    H_terms.append([stokes_op(stokes_phase), make_coeff_from_arr(S_half)])
    for k, op in enumerate(pump_ops):
        H_terms.append([op, make_coeff_from_arr(P_half_list[k])])

    args = {'tlist': tstep}
    result = mesolve(H_terms, rho0, tstep, c_ops, [], args=args)
    return result.states[-1]

def run_full_sequence(rho0, T, gamma):
    rho = rho0
    c_ops = get_spontaneous_emission_ops(gamma)
    for step_idx in range(4):
        rho = evolve_one_step_rho(rho, step_idx, T, c_ops)
    return rho

def average_fidelity(gamma_T, T_val):
    total_time = 40 * T_val
    gamma = gamma_T / total_time if gamma_T > 0 else 0.0
    fid_sum = 0.0

    for _ in range(n_samples):
        rand_vec = np.random.randn(4) + 1j * np.random.randn(4)
        rand_vec /= np.linalg.norm(rand_vec)

        psi_init = np.zeros(6, dtype=complex)
        psi_init[2:6] = rand_vec
        rho0 = Qobj(psi_init) * Qobj(psi_init).dag()

        psi_tgt_comp = QFT4 @ rand_vec
        psi_tgt_comp /= np.linalg.norm(psi_tgt_comp)
        psi_tgt_6d = np.zeros(6, dtype=complex)
        psi_tgt_6d[2:6] = psi_tgt_comp
        rho_target = Qobj(psi_tgt_6d) * Qobj(psi_tgt_6d).dag()

        rho_final = run_full_sequence(rho0, T_val, gamma)

        state_fid = np.real(expect(rho_target, rho_final))
        fid_sum += state_fid

    return fid_sum / n_samples

num_points = 20
gamma_T_values =[0]+ np.logspace(-6, -1, num_points)
fidelities = np.zeros(num_points)

print(f"正在对 T={T_val} 进行 quartit STIRAP 自发辐射扫描，抽取 {n_samples} 个随机态...")
start_time = time.time()

for idx, gT in enumerate(tqdm(gamma_T_values, desc="Scanning gamma*T")):
    fidelities[idx] = average_fidelity(gT, T_val)

total_time = time.time() - start_time
print(f"扫描完成，总用时 {total_time:.2f} 秒")

data = np.column_stack((gamma_T_values, fidelities))
header = "gammaT,AGF"
np.savetxt('quartit_STIRAP_spontaneous_emission.csv', data, delimiter=',', header=header, comments='')
print("数据已保存为 quartit_STIRAP_spontaneous_emission.csv")

plt.figure(figsize=(8, 6))
plt.plot(gamma_T_values, fidelities, 'o-', color='C1', linewidth=2,
         label=f'Quartit STIRAP (T={T_val})')
plt.xscale('log')
plt.xlabel(r'Dimensionless Spontaneous Emission ($\gamma t$)')
plt.ylabel('Average Gate Fidelity (AGF)')
plt.title(f'Effect of Spontaneous Emission on AGF (Monte Carlo over {n_samples} states)')
plt.grid(True, alpha=0.3, which='both')
plt.legend()
plt.tight_layout()
plt.savefig('quartit_STIRAP_emission_curve.png', dpi=200)
plt.show()

In [ ]:
import pandas as pd, numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
from matplotlib.ticker import AutoMinorLocator, LogLocator

lib='/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf'
libi='/usr/share/fonts/truetype/liberation/LiberationSans-Italic.ttf'
for f in (lib,libi): fm.fontManager.addfont(f)
AR=fm.FontProperties(fname=lib).get_name()
plt.rcParams.update({'font.family':'sans-serif','font.sans-serif':[AR,'Arial'],
    'axes.unicode_minus':False,'pdf.fonttype':42,'ps.fonttype':42,
    'mathtext.fontset':'custom','mathtext.rm':AR,'mathtext.it':AR+':italic',
    'mathtext.default':'it'})

df=pd.read_csv('all_protocols_scan.csv'); df.columns=[c.strip() for c in df.columns]
prot=df.iloc[:,0].astype(str).values
T =pd.to_numeric(df.iloc[:,1],errors='coerce').values
PK=pd.to_numeric(df.iloc[:,3],errors='coerce').values
def peak(p,t):
    m=(prot==p)&(np.isclose(T,t)); return PK[m][0] if m.any() else None
ts=sorted([t for t in set(T) if 1.5<=t<=30])
ex=np.array([100*(peak('quartit_alloptical',t)-peak('quartit_STIRSAP',t))
                 /peak('quartit_STIRSAP',t) for t in ts])
ts=np.array(ts)

CM=1/2.54; FS=16; AXLW=1.0; DLW=1.5; TKLEN=2.5
fig=plt.figure(figsize=(8.6*CM,6.7*CM))
ax=fig.add_axes([0.24,0.205,0.72,0.755])

ax.plot(ts,ex,'-',color='#1f4e79',lw=DLW,zorder=3)

ax.set_yscale('log')
ax.set_xlim(-1,31); ax.set_ylim(3e-2,7e2)
ax.set_xlabel(r'$\mathrm{T}$',fontsize=FS,labelpad=1)
ax.set_ylabel('Peak excess (%)',fontsize=FS,labelpad=1)

ax.set_xticks([0,10,20,30])
ax.yaxis.set_major_locator(LogLocator(base=10,numticks=4))
ax.set_yticks([1e-1,1e1,1e3]); ax.set_yticklabels([r'$10^{-1}$',r'$10^{1}$',r'$10^{3}$'])

ax.xaxis.set_minor_locator(AutoMinorLocator(5))
ax.yaxis.set_minor_locator(LogLocator(base=10,subs=np.arange(2,10)*0.1,numticks=100))

ax.tick_params(which='major',direction='in',length=TKLEN,width=AXLW,top=True,right=True,labelsize=FS)
ax.tick_params(which='minor',direction='in',length=TKLEN*0.6,width=AXLW,top=True,right=True)
for sp in ax.spines.values(): sp.set_linewidth(AXLW)

fig.savefig('peak_excess.pdf',dpi=300); fig.savefig('peak_excess_preview.png',dpi=300)
print('peak_excess done')
